# Imports

In [46]:
import re
import pandas as pd # Data manipulation/access
import utils # custom utility functions
from IPython.display import display, HTML # Hyperlink access
import fuzzywuzzy
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

#import PyInstaller # program containerization
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_colwidth', None)
#import PySimpleGUI as sg # User interface
#!pip install fuzzywuzzy #fuzzy match

In [29]:
!pip install fuzzywuzzy

# Data

In [2]:
# import all data
df = pd.read_excel('hsp_data.xlsx').fillna('')

# import findings and results in 2 separate dfs
findings_df = pd.read_excel('hsp_data.xlsx').iloc[:, list(range(0,20))].fillna('')
#results_df = pd.read_excel('hsp_data.xlsx').iloc[:, list(range(20,35))].fillna('')

In [6]:
# data preview
df[:2]

,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
0,D,,,,,,,,,agenesis,agenesis,,,,,,,,,hydrocephalus,SPG1,L1CAM,"X:153,861,516-153,886,173",https://www.ncbi.nlm.nih.gov/gene/3897,Complicated,XLR,NN,"1/300,000 Male",GROWTH\nHeight\n- Short stature (<5-15th perce...,"https://pubmed.ncbi.nlm.nih.gov/8305964/ , 10....",https://doi.org/10.1016/S0387-7604(97)00079-X,# 303350 https://www.omim.org/entry/303350,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,
1,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial le...",pons,present,,,SPG2,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,XLR,C,"<1/1,000,000",HEAD & NECK\nEyes\n- Nystagmus\n- Optic atroph...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?sea...,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,Multifocal - MS like lesions


# Data Cleaning

#### (Check utils.py)

# Making a Query

In [9]:
# Example of a query result: Frontal vs Posterior Predominance ==> posterior and Periventricular Involvement:Forceps Major ==> ear of lynx
df.loc[(df['Frontal vs Posterior Predominance']=='posterior')&(df['Periventricular Involvement:Forceps Major']=='ear of lynx')][df.iloc[:, list(range(20,35))].columns]

,Gene,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
1,SPG2,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,XLR,C,"<1/1,000,000",HEAD & NECK\nEyes\n- Nystagmus\n- Optic atroph...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?sea...,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,Multifocal - MS like lesions
16,SPG48,AP5Z1,"7:4,775,623-4,794,397",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AR,A,"<1 / 1,000,000",HEAD & NECK\nEyes\n- Retinopathy (adult onset ...,"https://doi.org/10.1186/s40035-019-0157-9 , ht...","https://doi.org/10.1093/brain/awu121, https://...",# 613647 https://omim.org/entry/613647?search=...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,


# Search Engine

#### Note:  

To check if a category has subcategories, check its $type$.  
If it's a $string$ then it has no subcategories else if it's a $dictionnary$ then it has subcategories.

```
>>> utils.check_multiple_dimensions('Subcortical Structures')
True
```

## Version 2
* Search logic: OR - Added filtering feature to select all rows with chosen value of interest present regardless of presence of additonal values.
* Keyboard interrupt feature (press enter)

In [16]:
def run_version2():
    
    # imports
    import pandas as pd # data manipulation/access
    import utils # custom utility functions
    from IPython.display import display, HTML # hyperlink access
    pd.set_option('display.max_columns', None)
    #pd.set_option('display.max_colwidth', None)

    dimension_1 = ['Diffuse or Multifocal','Symmetric','Frontal vs Posterior Predominance','Telltale Grey Matter Involvement','Cortical Involvement/Subcortical Lesions','U-Fiber/Juxtacortical Involvement', 'Ventriculomegaly vs Hydrocephalus']
    dimension_2 = ['Spinal Cord Involvement','Periventricular Involvement']
    dimension_3 = ['Subcortical Structures']

    # main loop
    # flag may be set to False depending on user input
    flag = True
    while flag:

        # data
        # as user adds search contraints, irrelevant rows are dropped
        # the dfs reset with every new search
        findings_df = pd.read_excel('hsp_data.xlsx').iloc[:, list(range(0,20))].fillna('')
        df = pd.read_excel('hsp_data.xlsx').fillna('')     

        # initially, additional_category is set to True to start a search
        # if set to False, return results with the only selected search category
        additional_category_flag = True
        while additional_category_flag:

            # display list of main categories
            access_categories = list(utils.findings_categorized.keys())
            print("List of categories:")
            print("*******************\n")
            print('\n\n'.join(access_categories))
            print()
            print()

            # user input for main category #
            category = input("Enter name of MRI finding: ")
            
            # keyboard interrupt
            if category == "":
                return "Forced exit. Goodbye!"
            
            # invalid input handling #
            while category not in access_categories:
                print()
                category = input("Invalid input. Choose a category from the list above (case sensitive): ")
            
            print()
            print("- You chose: [", category,"].")
            print()
            print()
            
            # check if main category is more than 1 dimension
            if category not in dimension_1:

                # 3 dimension
                if category not in dimension_2:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized['Subcortical Structures']
                    print("List of subcategories:")
                    print("**********************\n")
                    print('\n\n'.join(list(access_subcategories)))
                    print()
                    print()

                    # user input for subcategory #
                    subcategory = input("Choose a subcategory: ")

                    # keyboard interrupt
                    if subcategory == "":
                        return "Forced exit. Goodbye!"
                    
                    # invalid input handling #
                    while subcategory not in access_subcategories:
                        print()
                        subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")

                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # display list of sub-subcategories
                    access_sub_subcategories = access_subcategories[subcategory]
                    print("List of sub_subcategories:")
                    print("**************************\n")
                    print('\n\n'.join(list(access_sub_subcategories)))
                    print()
                    print()

                    # user input for sub_subcategory #
                    sub_subcategory = input("Choose a sub-subcategory: ")

                    # keyboard interrupt
                    if sub_subcategory == "":
                        return "Forced exit. Goodbye!"
                    
                    # invalid input handling #
                    while sub_subcategory not in access_sub_subcategories:
                        print()
                        sub_subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")
                        
                    print()
                    print("- You chose: [", sub_subcategory,"].")

                    # column title
                    col_title = str(category+':'+subcategory+':'+sub_subcategory)

                # 2 dimension
                else:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized[category]
                    print("\n\nList of subcategories:")
                    print("**********************\n")
                    print('\n\n'.join(list(access_subcategories)))
                    print()
                    print()

                    # user input for subcategory #
                    subcategory = input("Choose a subcategory: ")

                    # keyboard interrupt
                    if subcategory == "":
                        return "Forced exit. Goodbye!"
                    
                    # invalid input handling #
                    while subcategory not in access_subcategories:
                        print()
                        subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")

                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # column title
                    col_title = str(category+':'+subcategory)

            # 1 dimension        
            else:

                # column title
                col_title = category

            print()
            print()
            
            # # verify all unique values are selectable
            # column_values = list(df[col_title].unique())
            # access_values_of_interest = []
            # for column_value in column_values:
            #     values = column_value.split(',')
            #     for value in values:
            #         if str(value.strip()) not in access_values_of_interest:
            #             access_values_of_interest.append(value.strip())
            
            # display list of values of interest
            access_values_of_interest = findings_df[col_title].unique().tolist() # REPLACE #
            print("List of Values of Interest:")
            print("***************************\n")
            access_values_of_interest.remove('') if ('' in access_values_of_interest) else None
            print('\n\n'.join(access_values_of_interest))
            print()

            # user input for value of interest #
            value_of_interest = input("Choose a value of interest: ")

            # keyboard interrupt
            if value_of_interest == "":
                return "Forced exit. Goodbye!"
            
            # invalid input handling #
            while value_of_interest not in access_values_of_interest:
                print()
                value_of_interest = input("Invalid input. Choose a value of interest from the list above (case sensitive): ")

            print()
            print("- You chose: [", value_of_interest,"].")

            # keep relevant rows
            findings_df = findings_df.loc[(findings_df[col_title]==value_of_interest)]
            #findings_df = findings_df.loc[(findings_df[col_title]==value_of_interest)]

            # keep relevant rows
            #findings_df = findings_df.loc[(findings_df[col_title].str.contains(value_of_interest, case=False, na=False))]

            # prompt user to add additional search constraint #
            add_a_category = input("\n\nWould you to add another category? ")

            # keyboard interrupt
            if add_a_category == "":
                return "Forced exit. Goodbye!"

            if add_a_category == '0' or add_a_category.lower()=="no":
                additional_category_flag = False
        print()
        print()
        
        # Move "Gene" to first column
        current_columns = df.columns.tolist()
        df = df[['Gene'] + [col for col in current_columns if col != 'Gene']]
        
        # return results
        display(df.loc[findings_df.index])

        # prompt user to start new search #
        terminate = input("Would you like to search again? ")

        # keyboard interrupt
        if terminate == "":
            return "Forced exit. Goodbye!"

        if terminate == '0' or terminate.lower() == 'no':
            flag = False
        # data
        # as user adds search contraints, irrelevant rows are dropped
        # the dfs reset with every new search
        findings_df = pd.read_excel('hsp_data.xlsx').iloc[:, list(range(0,20))].fillna('')
        df = pd.read_excel('hsp_data.xlsx').fillna('')    
        
    # exit message
    print("Goodbye!")

### Example 1:

Diffuse or Multifocal: diffuse

Symmetric: asymmetric

Predominance: posterior


### Example 2: 

Diffuse or Multifocal: diffuse

Symmetric: symmetric

Cortical involvement: cortical atrophy

Periventricular: forceps minor, present

Subcortical: corpus callosum: genu, body, isthmus

Subcortical: posterior fossa: cerebellar, mild atrophy of vermis

### Example 3 

Ventriculomegaly vs Hydrocephalus

asymmetric ventriculomegaly + CSF



### Live example:

Diffuse or Multifocal: diffuse and multifocal

Symmetric: symmetric

Periventricular: forceps major: ear of lynx

Periventricular: forceps minor: present

Periventricular: other: present

Subcortical: corpus callosum: genu


### test

In [63]:
run_version2()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Diffuse or Multifocal



- You chose: [ Diffuse or Multifocal ].




List of Values of Interest:
***************************

D

D, M

M



Choose a value of interest:  D



- You chose: [ D ].


...


- You chose: [ posterior ].




Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
18,SPG50,D,asymmetric,posterior,,"global white matter atrophy, vascular tortuosity",,present,present,present,,,,present,present,short CC,present,,,,asymmetric ventriculomegaly + CSF issue,AP4M1,"7:100,100,794-100,109,039",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, NN","<1 / 1,000,000",HEAD & NECK\nHead\n- Microcephaly \nFace\n- Ps...,"10.3389/fneur.2018.01117 , 10.1002/ajmg.a.3651...","10.1093/brain/awz307 , 10.1212/WNL.00000000000...",# 612936 https://www.omim.org/entry/612936?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,


Would you like to search again?  no


Goodbye!


In [60]:
run_version2()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Diffuse or Multifocal



- You chose: [ Diffuse or Multifocal ].




List of Values of Interest:
***************************

D

D, M

M



Choose a value of interest:  D, M



- You chose: [ D, M ].


...


- You chose: [ present ].




Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
4,SPG5A,"D, M",symmetric,frontal,,"corona radiata, centrum smiovale",,ear of lynx,present,present,,present,,present,,,middle cerebellar peduncles,,present,,,CYP7B1,"8:64,586,575-64,798,737",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Both,AR,"C, AD","<1 / 1,000,000",HEAD & NECK:\nEars\n- Sensorineural hearing lo...,"https://doi.org/10.3174/ajnr.A7344 , 10.3389/f...","https://doi.org/10.1016/j.nmd.2008.10.009 , 10...",#270800 https://www.omim.org/entry/270800?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
16,SPG48,"D, M",symmetric,posterior,,subcortical lesions,,ear of lynx,present,present,,present,present,indenture,,,mild atrophy of cerebellum,,,,,AP5Z1,"7:4,775,623-4,794,397",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AR,A,"<1 / 1,000,000",HEAD & NECK\nEyes\n- Retinopathy (adult onset ...,"https://doi.org/10.1186/s40035-019-0157-9 , ht...","https://doi.org/10.1093/brain/awu121, https://...",# 613647 https://omim.org/entry/613647?search=...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,


Would you like to search again?  no


Goodbye!


### Run

In [57]:
run_version2()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Ventriculomegaly vs Hydrocephalus



- You chose: [ Ventriculomegaly vs Hydrocephalus ].




List of Values of Interest:
***************************

hydrocephalus

ventriculomegaly

asymmetric ventriculomegaly + CSF issue



Choose a value of interest:  asymmetric ventriculomegaly + CSF issue



- You chose: [ asymmetric ventriculomegaly + CSF issue ].




Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
18,SPG50,D,asymmetric,posterior,,"global white matter atrophy, vascular tortuosity",,present,present,present,,,,present,present,short CC,present,,,,asymmetric ventriculomegaly + CSF issue,AP4M1,"7:100,100,794-100,109,039",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, NN","<1 / 1,000,000",HEAD & NECK\nHead\n- Microcephaly \nFace\n- Ps...,"10.3389/fneur.2018.01117 , 10.1002/ajmg.a.3651...","10.1093/brain/awz307 , 10.1212/WNL.00000000000...",# 612936 https://www.omim.org/entry/612936?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,


Would you like to search again?  no


Goodbye!


### Testing Version 2

#### - Testing normal output

- Diffuse or Multifocal
- D

In [ ]:
run_version2()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Diffuse or Multifocal



- You chose: [ Diffuse or Multifocal ].




List of Values of Interest:
***************************

D

D, M

M



Choose a value of interest:  D



- You chose: [ D ].




Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
0,SPG1,D,,,,,,,,,agenesis,agenesis,,,,,,,,,hydrocephalus,L1CAM,"X:153,861,516-153,886,173",https://www.ncbi.nlm.nih.gov/gene/3897,Complicated,XLR,NN,"1/300,000 Male",GROWTH\nHeight\n- Short stature (<5-15th perce...,"https://pubmed.ncbi.nlm.nih.gov/8305964/ , 10....",https://doi.org/10.1016/S0387-7604(97)00079-X,# 303350 https://www.omim.org/entry/303350,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,
1,SPG2,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial le...",pons,present,,,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,XLR,C,"<1/1,000,000",HEAD & NECK\nEyes\n- Nystagmus\n- Optic atroph...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?sea...,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,Multifocal - MS like lesions
4,SPG5A,"D, M",symmetric,frontal,,"corona radiata, centrum smiovale",,ear of lynx,present,present,,present,,present,,,middle cerebellar peduncles,,present,,,CYP7B1,"8:64,586,575-64,798,737",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Both,AR,"C, AD","<1 / 1,000,000",HEAD & NECK:\nEars\n- Sensorineural hearing lo...,"https://doi.org/10.3174/ajnr.A7344 , 10.3389/f...","https://doi.org/10.1016/j.nmd.2008.10.009 , 10...",#270800 https://www.omim.org/entry/270800?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
5,SPG6,D,,,,,,,,,,,,,,,,,present,,,NIPA1,"15:22,786,225-22,829,789",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"C, AD, A","<1 / 1,000,000",GENITOURINARY\nBladder\n- Urinary urgency\n- U...,"10.3389/fneur.2018.01117 , https://doi.org/10....","https://doi.org/10.1007/s00234-005-1415-3 , ht...",# 600363 https://www.omim.org/entry/600363?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,Only 1 article,
6,SPG7,D,,,,,,,,,,,mild,,,,atrophy of cerebellum,,,,,PGN,"16:89,508,388-89,557,768",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Both,"AD, AR","C, AD, A, L","1-9/ 100,000",HEAD & NECK\nEyes\n- Optic atrophy\n- Supranuc...,"10.3389/fneur.2018.01117 , https://doi.org/10....",https://doi.org/10.3174/ajnr.A7017,#607259 https://www.omim.org/entry/607259?sear...,https://orpha.net/consor/cgi-bin/Disease_Searc...,,
7,SPG9B,D,symmetric,posterior,,,,present,present,present,,,,,present,,mild atrophy of vermis,,,,ventriculomegaly,ALDH18A1,"10:95,605,941-95,656,711",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, AD, A","<1 / 1,000,000",GROWTH\nHeight\n- Short stature\nOther\n- Grow...,"10.7860/JCDR/2021/49813.15390 , 10.3389/fneur....",https://doi.org/10.1016/j.braindev.2020.07.015,# 616586 https://www.omim.org/entry/616586?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,Not many articles w/ MRI's. Patient is 2.5 yea...,
8,SPG11,D,symmet

Would you like to search again?  no


Goodbye!


#### - Testing keyboard interrupt feature

In [ ]:
run_version2()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  


'Forced exit. Goodbye!'

#### - Testing invalid input handling (all levels)

In [ ]:
run_version2()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  goose


Invalid input. Choose a category from the list above (case sensitive):  wii


Invalid input. Choose a category from the list above (case sensitive):  Subcortical Structures



- You chose: [ Subcortical Structures ].


List of subcategories:
**********************

Corpus Callosum Involvement (Atrophy)

Posterior Fossa Involvement




...


- You chose: [ pons ].




Would you to add another category?  no


,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Oprha link,Periventricular Involvement:Other,Bracket notes
1,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial le...",pons,present,,,SPG2,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,XLR,C,"<1/1,000,000",HEAD & NECK\nEyes\n- Nystagmus\n- Optic atroph...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?sea...,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,Multifocal - MS like lesions


Would you like to search again?  no


Goodbye!


In [ ]:
# version 1
run_version1()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Periventricular Involvement



- You chose: [ Periventricular Involvement ].




List of subcategories:
**********************

Forceps Major

Forceps Minor

Other




Choose a subcategory:  Forceps Major



- You chose: [ Forceps Major ].




List of Values of Interest:
***************************

ear of lynx

present

ear of lynx, ears of grizzly



...


- You chose: [ ear of lynx ].




Would you to add another category?  no


,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Oprha link,Periventricular Involvement:Other,Bracket notes
1,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial le...",pons,present,,,SPG2,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,XLR,C,"<1/1,000,000",HEAD & NECK\nEyes\n- Nystagmus\n- Optic atroph...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?sea...,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,Multifocal - MS like lesions
4,"D, M",symmetric,frontal,,"corona radiata, centrum smiovale",,ear of lynx,present,present,,present,,present,,,middle cerebellar peduncles,,present,,,SPG5A,CYP7B1,"8:64,586,575-64,798,737",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Both,AR,"C, AD","<1 / 1,000,000",HEAD & NECK:\nEars\n- Sensorineural hearing lo...,"https://doi.org/10.3174/ajnr.A7344 , 10.3389/f...","https://doi.org/10.1016/j.nmd.2008.10.009 , 10...",#270800 https://www.omim.org/entry/270800?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
8,D,symmetric,frontal,,cortical atrophy,,ear of lynx,present,,present,present,present,present,,,mild atrophy of vermis,,,,ventriculomegaly,SPG11,,"15:44,562,696-44,663,662",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, C, AD, A","0.35/100,000",GROWTH\nWeight\n- Obesity\nHEAD & NECK\nEyes\n...,"https://doi.org/10.1002/mds.28519 , https://d...","10.3389/fneur.2018.01117 , https://doi.org/10....",# 604360 https://www.omim.org/entry/604360?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
9,D,symmetric,frontal,,,,ear of lynx,present,,,present,present,present,,,mild atrophy of vermis,,rare,,,SPG15,ZFYVE26,"14:67,728,892-67,816,590",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"C, AD, A","<1 / 1,000,000",HEAD & NECK\nEars\n- Hearing deficit\nEyes\n- ...,"https://doi.org/10.1016/j.ajhg.2008.03.004 , h...","10.3389/fneur.2018.01117 , https://doi.org/10....",# 270700 https://www.omim.org/entry/270700?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
16,"D, M",symmetric,posterior,,subcortical lesions,,ear of lynx,present,present,,present,present,indenture,,,mild atrophy of cerebellum,,,,,SPG48,AP5Z1,"7:4,775,623-4,794,397",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AR,A,"<1 / 1,000,000",HEAD & NECK\nEyes\n- Retinopathy (adult onset ...,"https://doi.org/10.1186/s40035-019-0157-9 , ht...","https://doi.org/10.1093/brain/awu121, https://...",# 613647 https://omim.org/entry/613647?search=...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
17,D,symmetric,,globus pallida calcifications,,,ear of lynx,present,,,,present,,,,,,,,,SPG49,TECPR2,,,Complicated,AR,I,"<1 / 1,000,000",,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.ajhg.2012.11.001,,https://www.orpha.net

Would you like to search again?  no


Goodbye!


In [ ]:
# version 2
run_version2()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Periventricular Involvement



- You chose: [ Periventricular Involvement ].




List of subcategories:
**********************

Forceps Major

Forceps Minor

Other




Choose a subcategory:  Forceps Major



- You chose: [ Forceps Major ].




List of Values of Interest:
***************************

ear of lynx

present

ear of lynx, ears of grizzly



...


- You chose: [ ear of lynx ].




Would you to add another category?  no


,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Oprha link,Periventricular Involvement:Other,Bracket notes
1,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial le...",pons,present,,,SPG2,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,XLR,C,"<1/1,000,000",HEAD & NECK\nEyes\n- Nystagmus\n- Optic atroph...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?sea...,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,Multifocal - MS like lesions
4,"D, M",symmetric,frontal,,"corona radiata, centrum smiovale",,ear of lynx,present,present,,present,,present,,,middle cerebellar peduncles,,present,,,SPG5A,CYP7B1,"8:64,586,575-64,798,737",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Both,AR,"C, AD","<1 / 1,000,000",HEAD & NECK:\nEars\n- Sensorineural hearing lo...,"https://doi.org/10.3174/ajnr.A7344 , 10.3389/f...","https://doi.org/10.1016/j.nmd.2008.10.009 , 10...",#270800 https://www.omim.org/entry/270800?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
8,D,symmetric,frontal,,cortical atrophy,,ear of lynx,present,,present,present,present,present,,,mild atrophy of vermis,,,,ventriculomegaly,SPG11,,"15:44,562,696-44,663,662",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, C, AD, A","0.35/100,000",GROWTH\nWeight\n- Obesity\nHEAD & NECK\nEyes\n...,"https://doi.org/10.1002/mds.28519 , https://d...","10.3389/fneur.2018.01117 , https://doi.org/10....",# 604360 https://www.omim.org/entry/604360?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
9,D,symmetric,frontal,,,,ear of lynx,present,,,present,present,present,,,mild atrophy of vermis,,rare,,,SPG15,ZFYVE26,"14:67,728,892-67,816,590",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"C, AD, A","<1 / 1,000,000",HEAD & NECK\nEars\n- Hearing deficit\nEyes\n- ...,"https://doi.org/10.1016/j.ajhg.2008.03.004 , h...","10.3389/fneur.2018.01117 , https://doi.org/10....",# 270700 https://www.omim.org/entry/270700?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
15,D,symmetric and asymmetric,posterior,,"cortical malformation, BPP",,"ear of lynx, ears of grizzly",ears of grizzly,heterotopia,,,present,severe atrophy,severe atrophy,"short CC, hemigenesis",mild atrophy of vermis,,,,ventriculomegaly,SPG47,AP4B1,"1:113,894,194-113,905,028",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, NN","<1 / 1,000,000",GROWTH\nHeight\n- Short stature\nHEAD & NECK\n...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...","10.1093/brain/awz307 , https://doi.org/10.1002...",# 614066 https://www.omim.org/entry/614066?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
16,"D, M",symmetric,posterior,,subcortical lesions,,ear of lynx,present,present,,present,present,indenture,,,mild atrophy of cerebellum,,,,,

Would you like to search again?  no


Goodbye!


## Version 1

 - Search logic: AND

In [4]:
def run_version1():
    
    # imports
    import pandas as pd # data manipulation/access
    import utils # custom utility functions
    from IPython.display import display, HTML # hyperlink access
    #pd.set_option('display.max_columns', None)
    #pd.set_option('display.max_colwidth', None)
    
    
    dimension_1 = ['Diffuse or Multifocal','Symmetric','Frontal vs Posterior Predominance','Telltale Grey Matter Involvement','Cortical Involvement/Subcortical Lesions','U-Fiber/Juxtacortical Involvement', 'Ventriculomegaly vs Hydrocephalus']
    dimension_2 = ['Spinal Cord Involvement','Periventricular Involvement']
    dimension_3 = ['Subcortical Structures']

    # main loop
    # flag may be set to False depending on user input
    flag = True
    while flag:
        
        # data
        # as user adds search contraints, irrelevant rows are dropped
        # the dfs reset with every new search
        findings_df = pd.read_excel('hsp_data.xlsx').iloc[:, list(range(0,20))].fillna('')
        df = pd.read_excel('hsp_data.xlsx').fillna('')
        
        # initially, additional_category is set to True to start a search
        # if set to False, return results with the only selected search category
        additional_category_flag = True
        while additional_category_flag:
            
            # display list of main categories
            access_categories = list(utils.findings_categorized.keys())
            print("List of categories:")
            print("*******************\n")
            print('\n\n'.join(access_categories))
            print()
            print()

            # user input for main category
            category = input("Enter name of MRI finding: ")
            print()
            print("- You chose: [", category,"].")
            print()
            print()
            # check if main category is more than 1 dimension
            if category not in dimension_1:

                # 3 dimension
                if category not in dimension_2:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized['Subcortical Structures']
                    print("List of subcategories:")
                    print("**********************\n")
                    print('\n\n'.join(list(access_subcategories)))
                    print()
                    print()

                    # user input for subcategory
                    subcategory = input("Choose a subcategory: ")
                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # display list of sub-subcategories
                    access_sub_subcategories = access_subcategories[subcategory]
                    print("List of sub_subcategories:")
                    print("**************************\n")
                    print('\n\n'.join(list(access_sub_subcategories)))
                    print()
                    print()

                    # user input for sub_subcategory
                    sub_subcategory = input("Choose a sub_subcategory: ")
                    print()
                    print("- You chose: [", sub_subcategory,"].")

                    # column title
                    col_title = str(category+':'+subcategory+':'+sub_subcategory)

                # 2 dimension
                else:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized[category]
                    print("\n\nList of subcategories:")
                    print("**********************\n")
                    print('\n\n'.join(list(access_subcategories)))
                    print()
                    print()

                    # user input for subcategory
                    subcategory = input("Choose a subcategory: ")
                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # column title
                    col_title = str(category+':'+subcategory)

            # 1 dimension        
            else:

                # column title
                col_title = category

            print()
            print()

            # display list of values of interest
            access_values_of_interest = findings_df[col_title].unique().tolist()
            print("List of Values of Interest:")
            print("***************************\n")
            access_values_of_interest.remove('') if ('' in access_values_of_interest) else None
            print('\n\n'.join(access_values_of_interest))
            print()

            # user input for value of interest
            value_of_interest = input("Choose a value of interest: ")
            print()
            print("- You chose: [", value_of_interest,"].")

            # keep relevant rows
            findings_df = findings_df.loc[(findings_df[col_title]==value_of_interest)]

            # prompt user to add additional search constraint
            add_a_category = input("\n\nWould you to add another category? ")
            if add_a_category == '0' or add_a_category.lower()=="no":
                additional_category_flag = False
        print()
        print()

        # return results
        display(df.loc[findings_df.index])

        # terminate/new search
        terminate = input("Would you like to search again? ")
        if terminate == '0' or terminate.lower() == 'no':
            flag = False

    # exit message
    print("Goodbye!")

### Testing Version 1

#### - Testing normal output
* 1st category
- Spinal Cord Involvement
- Spinal Cord Atrophy
- present
* 2nd category
- Frontal vs Posterior Predominance
- posterior

In [42]:
run_version1()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Spinal Cord Involvement



- You chose: [ Spinal Cord Involvement ].




List of subcategories:
**********************

Spinal Cord Atrophy

Abnormal Signal/White Matter Tract




Choose a subcategory:  Spinal Cord Atrophy



- You chose: [ Spinal Cord Atrophy ].




List of Values of Interest:
***************************

present

mild

rare



...


- You chose: [ posterior ].




Would you to add another category?  no


,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
1,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial le...",pons,present,,,SPG2,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,XLR,C,"<1/1,000,000",HEAD & NECK\nEyes\n- Nystagmus\n- Optic atroph...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?sea...,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,Multifocal - MS like lesions
3,,symmetric,posterior,,,,,,present,present,,,,,,atrophy of cerebellum,,present,,ventriculomegaly,SPG4,SPAST,"2:32,063,556-32,157,637+6:35",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"I, C, AD, A","0.9/100,00",HEAD & NECK\nEyes\n- Nystagmus (rare)\nGENITOU...,"10.1186/s12883-022-02595-4 , https://doi.org/...",https://doi.org/10.1007/s00234-005-1415-3; 10....,#182601 https://www.omim.org/entry/182601?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
14,D,,posterior,,,,,,present,,,present,present,,,atrophy of vermis,,present,,ventriculomegaly,SPG46,GBA2,"9:35,736,866-35,749,228",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, C","<1 / 1,000,000",HEAD & NECK\nEars\n- Hearing loss (in some pat...,"10.3389/fneur.2018.01117 , 10.1016/j.ajhg.2012...","https://doi.org/10.1007/s10048-010-0249-2 , 10...",# 614409 https://www.omim.org/entry/614409?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,


Would you like to search again?  no


Goodbye!


#### - Testing normal output
* 1st category
- Ventriculomegaly vs Hydrocephalus
- ventriculomegaly

* 2nd category
- Symmetric
- symmetric

* 3rd category
- Periventricular Involvement
- Forceps Major
- present

In [43]:
run_version1()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Ventriculomegaly vs Hydrocephalus



- You chose: [ Ventriculomegaly vs Hydrocephalus ].




List of Values of Interest:
***************************

hydrocephalus

ventriculomegaly

asymmetric ventriculomegaly + CSF issue



Choose a value of interest:  ventriculomegaly



- You chose: [ ventriculomegaly ].


...


- You chose: [ present ].




Would you to add another category?  no


,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
7,D,symmetric,posterior,,,,present,present,present,,,,,present,,mild atrophy of vermis,,,,ventriculomegaly,SPG9B,ALDH18A1,"10:95,605,941-95,656,711",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, AD, A","<1 / 1,000,000",GROWTH\nHeight\n- Short stature\nOther\n- Grow...,"10.7860/JCDR/2021/49813.15390 , 10.3389/fneur....",https://doi.org/10.1016/j.braindev.2020.07.015,# 616586 https://www.omim.org/entry/616586?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,Not many articles w/ MRI's. Patient is 2.5 yea...,
19,"D, M",symmetric,,,"global white matter atrophy, BPP",hypomyelination,present,,,,,,,,thin CC,,,,,ventriculomegaly,SPG52,AP4S1,"14:31,025,106-31,096,450",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, NN","<1 / 1,000,000",GROWTH\nHeight\n- Short stature\nHEAD & NECK\n...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...","10.1093/brain/awz307 , 10.1212/WNL.00000000000...",# 614067 https://www.omim.org/entry/614067?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,


Would you like to search again?  no


Goodbye!


## BUG (main)

#### - test1() works with :
``` python
# keep relevant rows
            findings_df = findings_df.loc[(findings_df[col_title].str.contains(value_of_interest, case=False, na=False))]
```

In [ ]:
def test1():
    
    # imports
    import pandas as pd # data manipulation/access
    import utils # custom utility functions
    from IPython.display import display, HTML # hyperlink access
    pd.set_option('display.max_columns', None)
    #pd.set_option('display.max_colwidth', None)

    dimension_1 = ['Diffuse or Multifocal','Symmetric','Frontal vs Posterior Predominance','Telltale Grey Matter Involvement','Cortical Involvement/Subcortical Lesions','U-Fiber/Juxtacortical Involvement', 'Ventriculomegaly vs Hydrocephalus']
    dimension_2 = ['Spinal Cord Involvement','Periventricular Involvement']
    dimension_3 = ['Subcortical Structures']

    # main loop
    # flag may be set to False depending on user input
    flag = True
    while flag:

        # data
        # as user adds search contraints, irrelevant rows are dropped
        # the dfs reset with every new search
        findings_df = pd.read_excel('hsp_data.xlsx').iloc[:, list(range(0,20))].fillna('')
        df = pd.read_excel('hsp_data.xlsx').fillna('')     

        # initially, additional_category is set to True to start a search
        # if set to False, return results with the only selected search category
        additional_category_flag = True
        while additional_category_flag:

            # display list of main categories
            access_categories = list(utils.findings_categorized.keys())
            print("List of categories:")
            print("*******************\n")
            print('\n\n'.join(access_categories))
            print()
            print()

            # user input for main category #
            category = input("Enter name of MRI finding: ")
            
            # keyboard interrupt
            if category == "":
                return "Forced exit."
            
            # invalid input handling #
            while category not in access_categories:
                print()
                category = input("Invalid input. Choose a category from the list above (case sensitive): ")
            
            print()
            print("- You chose: [", category,"].")
            print()
            print()
            
            # check if main category is more than 1 dimension
            if category not in dimension_1:

                # 3 dimension
                if category not in dimension_2:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized['Subcortical Structures']
                    print("List of subcategories:")
                    print("**********************\n")
                    print('\n\n'.join(list(access_subcategories)))
                    print()
                    print()

                    # user input for subcategory #
                    subcategory = input("Choose a subcategory: ")

                    # keyboard interrupt
                    if subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while subcategory not in access_subcategories:
                        print()
                        subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")

                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # display list of sub-subcategories
                    access_sub_subcategories = access_subcategories[subcategory]
                    print("List of sub_subcategories:")
                    print("**************************\n")
                    print('\n\n'.join(list(access_sub_subcategories)))
                    print()
                    print()

                    # user input for sub_subcategory #
                    sub_subcategory = input("Choose a sub-subcategory: ")

                    # keyboard interrupt
                    if sub_subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while sub_subcategory not in access_sub_subcategories:
                        print()
                        sub_subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")
                        
                    print()
                    print("- You chose: [", sub_subcategory,"].")

                    # column title
                    col_title = str(category+':'+subcategory+':'+sub_subcategory)

                # 2 dimension
                else:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized[category]
                    print("\n\nList of subcategories:")
                    print("**********************\n")
                    print('\n\n'.join(list(access_subcategories)))
                    print()
                    print()

                    # user input for subcategory #
                    subcategory = input("Choose a subcategory: ")

                    # keyboard interrupt
                    if subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while subcategory not in access_subcategories:
                        print()
                        subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")

                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # column title
                    col_title = str(category+':'+subcategory)

            # 1 dimension        
            else:

                # column title
                col_title = category
            print()
            print()
            
            
            # display list of values of interest
            access_values_of_interest = findings_df[col_title].unique().tolist() # REPLACE #
            print("List of Values of Interest:")
            print("***************************\n")
            access_values_of_interest.remove('') if ('' in access_values_of_interest) else None
            print('\n\n'.join(access_values_of_interest))
            print()

            # user input for value of interest #
            value_of_interest = input("Choose a value of interest: ")

            # keyboard interrupt
            if value_of_interest == "":
                return "Forced exit."
            
            # invalid input handling #
            while value_of_interest not in access_values_of_interest:
                print()
                value_of_interest = input("Invalid input. Choose a value of interest from the list above (case sensitive): ")

            print()
            print("- You chose: [", value_of_interest,"].")

            # keep relevant rows
            findings_df = findings_df.loc[(findings_df[col_title].str.contains(value_of_interest, case=False, na=False))]

            # prompt user to add additional search constraint #
            add_a_category = input("\n\nWould you to add another category? ")

            # keyboard interrupt
            if add_a_category == "":
                return "Forced exit."

            if add_a_category == '0' or add_a_category.lower()=="no":
                additional_category_flag = False
        print()
        print()
        
        # move "Gene" to first column
        current_columns = df.columns.tolist()
        df = df[['Gene'] + [col for col in current_columns if col != 'Gene']]
        
        # return results
        display(df.loc[findings_df.index])

        # prompt user to start new search #
        terminate = input("Would you like to search again? ")
        if terminate == '0' or terminate.lower() == 'no' or terminate == "":
            flag = False

    # exit message
    print("Goodbye!")

In [ ]:
test1()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Diffuse or Multifocal



- You chose: [ Diffuse or Multifocal ].




List of Values of Interest:
***************************

D

D, M

M



Choose a value of interest:  M



- You chose: [ M ].


...


- You chose: [ D ].




Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
0,SPG1,D,,,,,,,,,agenesis,agenesis,,,,,,,,,hydrocephalus,L1CAM,"X:153,861,516-153,886,173",https://www.ncbi.nlm.nih.gov/gene/3897,Complicated,XLR,NN,"1/300,000 Male",GROWTH\nHeight\n- Short stature (<5-15th perce...,"https://pubmed.ncbi.nlm.nih.gov/8305964/ , 10....",https://doi.org/10.1016/S0387-7604(97)00079-X,# 303350 https://www.omim.org/entry/303350,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,
1,SPG2,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial le...",pons,present,,,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,XLR,C,"<1/1,000,000",HEAD & NECK\nEyes\n- Nystagmus\n- Optic atroph...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?sea...,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,Multifocal - MS like lesions
4,SPG5A,"D, M",symmetric,frontal,,"corona radiata, centrum smiovale",,ear of lynx,present,present,,present,,present,,,middle cerebellar peduncles,,present,,,CYP7B1,"8:64,586,575-64,798,737",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Both,AR,"C, AD","<1 / 1,000,000",HEAD & NECK:\nEars\n- Sensorineural hearing lo...,"https://doi.org/10.3174/ajnr.A7344 , 10.3389/f...","https://doi.org/10.1016/j.nmd.2008.10.009 , 10...",#270800 https://www.omim.org/entry/270800?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
5,SPG6,D,,,,,,,,,,,,,,,,,present,,,NIPA1,"15:22,786,225-22,829,789",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"C, AD, A","<1 / 1,000,000",GENITOURINARY\nBladder\n- Urinary urgency\n- U...,"10.3389/fneur.2018.01117 , https://doi.org/10....","https://doi.org/10.1007/s00234-005-1415-3 , ht...",# 600363 https://www.omim.org/entry/600363?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,Only 1 article,
6,SPG7,D,,,,,,,,,,,mild,,,,atrophy of cerebellum,,,,,PGN,"16:89,508,388-89,557,768",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Both,"AD, AR","C, AD, A, L","1-9/ 100,000",HEAD & NECK\nEyes\n- Optic atrophy\n- Supranuc...,"10.3389/fneur.2018.01117 , https://doi.org/10....",https://doi.org/10.3174/ajnr.A7017,#607259 https://www.omim.org/entry/607259?sear...,https://orpha.net/consor/cgi-bin/Disease_Searc...,,
7,SPG9B,D,symmetric,posterior,,,,present,present,present,,,,,present,,mild atrophy of vermis,,,,ventriculomegaly,ALDH18A1,"10:95,605,941-95,656,711",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, AD, A","<1 / 1,000,000",GROWTH\nHeight\n- Short stature\nOther\n- Grow...,"10.7860/JCDR/2021/49813.15390 , 10.3389/fneur....",https://doi.org/10.1016/j.braindev.2020.07.015,# 616586 https://www.omim.org/entry/616586?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,Not many articles w/ MRI's. Patient is 2.5 yea...,
8,SPG11,D,symmet

Would you like to search again?  no


Goodbye!


#### - test2() works with :
``` python
# verify all unique values are selectable
            column_values = list(df[col_title].unique())
            access_values_of_interest = []
            for column_value in column_values:
                values = column_value.split(',')
                for value in values:
                    if str(value.strip()) not in access_values_of_interest:
                        access_values_of_interest.append(value.strip())
```

In [ ]:
def test2():
    
    # imports
    import pandas as pd # data manipulation/access
    import utils # custom utility functions
    from IPython.display import display, HTML # hyperlink access
    pd.set_option('display.max_columns', None)
    #pd.set_option('display.max_colwidth', None)

    dimension_1 = ['Diffuse or Multifocal','Symmetric','Frontal vs Posterior Predominance','Telltale Grey Matter Involvement','Cortical Involvement/Subcortical Lesions','U-Fiber/Juxtacortical Involvement', 'Ventriculomegaly vs Hydrocephalus']
    dimension_2 = ['Spinal Cord Involvement','Periventricular Involvement']
    dimension_3 = ['Subcortical Structures']

    # main loop
    # flag may be set to False depending on user input
    flag = True
    while flag:

        # data
        # as user adds search contraints, irrelevant rows are dropped
        # the dfs reset with every new search
        findings_df = pd.read_excel('hsp_data.xlsx').iloc[:, list(range(0,20))].fillna('')
        df = pd.read_excel('hsp_data.xlsx').fillna('')     

        # initially, additional_category is set to True to start a search
        # if set to False, return results with the only selected search category
        additional_category_flag = True
        while additional_category_flag:

            # display list of main categories
            access_categories = list(utils.findings_categorized.keys())
            print("List of categories:")
            print("*******************\n")
            print('\n\n'.join(access_categories))
            print()
            print()

            # user input for main category #
            category = input("Enter name of MRI finding: ")
            
            # keyboard interrupt
            if category == "":
                return "Forced exit."
            
            # invalid input handling #
            while category not in access_categories:
                print()
                category = input("Invalid input. Choose a category from the list above (case sensitive): ")
            
            print()
            print("- You chose: [", category,"].")
            print()
            print()
            
            # check if main category is more than 1 dimension
            if category not in dimension_1:

                # 3 dimension
                if category not in dimension_2:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized['Subcortical Structures']
                    print("List of subcategories:")
                    print("**********************\n")
                    print('\n\n'.join(list(access_subcategories)))
                    print()
                    print()

                    # user input for subcategory #
                    subcategory = input("Choose a subcategory: ")

                    # keyboard interrupt
                    if subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while subcategory not in access_subcategories:
                        print()
                        subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")

                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # display list of sub-subcategories
                    access_sub_subcategories = access_subcategories[subcategory]
                    print("List of sub_subcategories:")
                    print("**************************\n")
                    print('\n\n'.join(list(access_sub_subcategories)))
                    print()
                    print()

                    # user input for sub_subcategory #
                    sub_subcategory = input("Choose a sub-subcategory: ")

                    # keyboard interrupt
                    if sub_subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while sub_subcategory not in access_sub_subcategories:
                        print()
                        sub_subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")
                        
                    print()
                    print("- You chose: [", sub_subcategory,"].")

                    # column title
                    col_title = str(category+':'+subcategory+':'+sub_subcategory)

                # 2 dimension
                else:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized[category]
                    print("\n\nList of subcategories:")
                    print("**********************\n")
                    print('\n\n'.join(list(access_subcategories)))
                    print()
                    print()

                    # user input for subcategory #
                    subcategory = input("Choose a subcategory: ")

                    # keyboard interrupt
                    if subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while subcategory not in access_subcategories:
                        print()
                        subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")

                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # column title
                    col_title = str(category+':'+subcategory)

            # 1 dimension        
            else:

                # column title
                col_title = category
            print()
            print()
            
            # verify all unique values are selectable
            column_values = list(df[col_title].unique())
            access_values_of_interest = []
            for column_value in column_values:
                values = column_value.split(',')
                for value in values:
                    if str(value.strip()) not in access_values_of_interest:
                        access_values_of_interest.append(value.strip())

            # display list of values of interest
            print("List of Values of Interest:")
            print("***************************\n")
            access_values_of_interest.remove('') if ('' in access_values_of_interest) else None
            print('\n\n'.join(access_values_of_interest))
            print()

            # user input for value of interest #
            value_of_interest = input("Choose a value of interest: ")

            # keyboard interrupt
            if value_of_interest == "":
                return "Forced exit."
            
            # invalid input handling #
            while value_of_interest not in access_values_of_interest:
                print()
                value_of_interest = input("Invalid input. Choose a value of interest from the list above (case sensitive): ")

            print()
            print("- You chose: [", value_of_interest,"].")

            # keep relevant rows
            findings_df = findings_df.loc[(findings_df[col_title].str.contains(value_of_interest, case=False, na=False))]

            # prompt user to add additional search constraint #
            add_a_category = input("\n\nWould you to add another category? ")

            # keyboard interrupt
            if add_a_category == "":
                return "Forced exit."

            if add_a_category == '0' or add_a_category.lower()=="no":
                additional_category_flag = False
        print()
        print()
        
        # move "Gene" to first column
        current_columns = df.columns.tolist()
        df = df[['Gene'] + [col for col in current_columns if col != 'Gene']]
        
        # return results
        display(df.loc[findings_df.index])

        # prompt user to start new search #
        terminate = input("Would you like to search again? ")
        if terminate == '0' or terminate.lower() == 'no' or terminate == "":
            flag = False

    # exit message
    print("Goodbye!")

In [ ]:
test2()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Diffuse or Multifocal



- You chose: [ Diffuse or Multifocal ].




List of Values of Interest:
***************************

D

M



Choose a value of interest:  D



- You chose: [ D ].


...


- You chose: [ posterior ].




Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
1,SPG2,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial le...",pons,present,,,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,XLR,C,"<1/1,000,000",HEAD & NECK\nEyes\n- Nystagmus\n- Optic atroph...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?sea...,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,Multifocal - MS like lesions
15,SPG47,D,symmetric and asymmetric,posterior,,"cortical malformation, BPP",,"ear of lynx, ears of grizzly",ears of grizzly,heterotopia,,,present,severe atrophy,severe atrophy,"short CC, hemigenesis",mild atrophy of vermis,,,,ventriculomegaly,AP4B1,"1:113,894,194-113,905,028",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, NN","<1 / 1,000,000",GROWTH\nHeight\n- Short stature\nHEAD & NECK\n...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...","10.1093/brain/awz307 , https://doi.org/10.1002...",# 614066 https://www.omim.org/entry/614066?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
18,SPG50,D,asymmetric,posterior,,"global white matter atrophy, vascular tortuosity",,present,present,present,,,,present,present,short CC,present,,,,asymmetric ventriculomegaly + CSF issue,AP4M1,"7:100,100,794-100,109,039",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, NN","<1 / 1,000,000",HEAD & NECK\nHead\n- Microcephaly \nFace\n- Ps...,"10.3389/fneur.2018.01117 , 10.1002/ajmg.a.3651...","10.1093/brain/awz307 , 10.1212/WNL.00000000000...",# 612936 https://www.omim.org/entry/612936?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,


Would you like to search again?  no


Goodbye!


#### - test3() does NOT with :
``` python
# keep relevant rows
            findings_df = findings_df.loc[(findings_df[col_title].str.startswith(value_of_interest))&(findings_df[col_title].str.contains(value_of_interest, case=False, na=False))]

```

In [ ]:
def test3():
    
    # imports
    import pandas as pd # data manipulation/access
    import utils # custom utility functions
    from IPython.display import display, HTML # hyperlink access
    pd.set_option('display.max_columns', None)
    #pd.set_option('display.max_colwidth', None)

    dimension_1 = ['Diffuse or Multifocal','Symmetric','Frontal vs Posterior Predominance','Telltale Grey Matter Involvement','Cortical Involvement/Subcortical Lesions','U-Fiber/Juxtacortical Involvement', 'Ventriculomegaly vs Hydrocephalus']
    dimension_2 = ['Spinal Cord Involvement','Periventricular Involvement']
    dimension_3 = ['Subcortical Structures']

    # main loop
    # flag may be set to False depending on user input
    flag = True
    while flag:

        # data
        # as user adds search contraints, irrelevant rows are dropped
        # the dfs reset with every new search
        findings_df = pd.read_excel('hsp_data.xlsx').iloc[:, list(range(0,20))].fillna('')
        df = pd.read_excel('hsp_data.xlsx').fillna('')     

        # initially, additional_category is set to True to start a search
        # if set to False, return results with the only selected search category
        additional_category_flag = True
        while additional_category_flag:

            # display list of main categories
            access_categories = list(utils.findings_categorized.keys())
            print("List of categories:")
            print("*******************\n")
            print('\n\n'.join(access_categories))
            print()
            print()

            # user input for main category #
            category = input("Enter name of MRI finding: ")
            
            # keyboard interrupt
            if category == "":
                return "Forced exit."
            
            # invalid input handling #
            while category not in access_categories:
                print()
                category = input("Invalid input. Choose a category from the list above (case sensitive): ")
            
            print()
            print("- You chose: [", category,"].")
            print()
            print()
            
            # check if main category is more than 1 dimension
            if category not in dimension_1:

                # 3 dimension
                if category not in dimension_2:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized['Subcortical Structures']
                    print("List of subcategories:")
                    print("**********************\n")
                    print('\n\n'.join(list(access_subcategories)))
                    print()
                    print()

                    # user input for subcategory #
                    subcategory = input("Choose a subcategory: ")

                    # keyboard interrupt
                    if subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while subcategory not in access_subcategories:
                        print()
                        subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")

                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # display list of sub-subcategories
                    access_sub_subcategories = access_subcategories[subcategory]
                    print("List of sub_subcategories:")
                    print("**************************\n")
                    print('\n\n'.join(list(access_sub_subcategories)))
                    print()
                    print()

                    # user input for sub_subcategory #
                    sub_subcategory = input("Choose a sub-subcategory: ")

                    # keyboard interrupt
                    if sub_subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while sub_subcategory not in access_sub_subcategories:
                        print()
                        sub_subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")
                        
                    print()
                    print("- You chose: [", sub_subcategory,"].")

                    # column title
                    col_title = str(category+':'+subcategory+':'+sub_subcategory)

                # 2 dimension
                else:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized[category]
                    print("\n\nList of subcategories:")
                    print("**********************\n")
                    print('\n\n'.join(list(access_subcategories)))
                    print()
                    print()

                    # user input for subcategory #
                    subcategory = input("Choose a subcategory: ")

                    # keyboard interrupt
                    if subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while subcategory not in access_subcategories:
                        print()
                        subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")

                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # column title
                    col_title = str(category+':'+subcategory)

            # 1 dimension        
            else:

                # column title
                col_title = category
            print()
            print()
            
            
            # display list of values of interest
            access_values_of_interest = findings_df[col_title].unique().tolist() # REPLACE #
            print("List of Values of Interest:")
            print("***************************\n")
            access_values_of_interest.remove('') if ('' in access_values_of_interest) else None
            print('\n\n'.join(access_values_of_interest))
            print()

            # user input for value of interest #
            value_of_interest = input("Choose a value of interest: ")

            # keyboard interrupt
            if value_of_interest == "":
                return "Forced exit."
            
            # invalid input handling #
            while value_of_interest not in access_values_of_interest:
                print()
                value_of_interest = input("Invalid input. Choose a value of interest from the list above (case sensitive): ")

            print()
            print("- You chose: [", value_of_interest,"].")

            # keep relevant rows
            findings_df = findings_df.loc[(findings_df[col_title].str.startswith(value_of_interest))&(findings_df[col_title].str.contains(value_of_interest, case=False, na=False))]

            # prompt user to add additional search constraint #
            add_a_category = input("\n\nWould you to add another category? ")

            # keyboard interrupt
            if add_a_category == "":
                return "Forced exit."

            if add_a_category == '0' or add_a_category.lower()=="no":
                additional_category_flag = False
        print()
        print()
        
        # move "Gene" to first column
        current_columns = df.columns.tolist()
        df = df[['Gene'] + [col for col in current_columns if col != 'Gene']]
        
        # return results
        display(df.loc[findings_df.index])

        # prompt user to start new search #
        terminate = input("Would you like to search again? ")
        if terminate == '0' or terminate.lower() == 'no' or terminate == "":
            flag = False

    # exit message
    print("Goodbye!")

In [ ]:
test3()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Diffuse or Multifocal



- You chose: [ Diffuse or Multifocal ].




List of Values of Interest:
***************************

D

D, M

M



Choose a value of interest:  D



- You chose: [ D ].


...


- You chose: [ posterior ].




Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
1,SPG2,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial le...",pons,present,,,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,XLR,C,"<1/1,000,000",HEAD & NECK\nEyes\n- Nystagmus\n- Optic atroph...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?sea...,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,Multifocal - MS like lesions
18,SPG50,D,asymmetric,posterior,,"global white matter atrophy, vascular tortuosity",,present,present,present,,,,present,present,short CC,present,,,,asymmetric ventriculomegaly + CSF issue,AP4M1,"7:100,100,794-100,109,039",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, NN","<1 / 1,000,000",HEAD & NECK\nHead\n- Microcephaly \nFace\n- Ps...,"10.3389/fneur.2018.01117 , 10.1002/ajmg.a.3651...","10.1093/brain/awz307 , 10.1212/WNL.00000000000...",# 612936 https://www.omim.org/entry/612936?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,


Would you like to search again?  


Goodbye!


## Testing Main

#### - General output test
- Subcortical Structures
- Corpus Callosum Involvement (Atrophy)
- Splenium
- severe atrophy

In [64]:
main()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Subcortical Structures



- You chose: [ Subcortical Structures ].


List of subcategories:
**********************

Corpus Callosum Involvement (Atrophy)

Posterior Fossa Involvement




Choose a subcategory:  Corpus Callosum Involvement (Atrophy)



- You chose: [ Corpus Callosum Involvement (Atrophy) ].


List of sub_subcategories:
**************************

Rostrum

Genu

Body

Isthmus

Splenium

Other




...


- You chose: [ severe atrophy ].




Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
15,SPG47,D,symmetric and asymmetric,posterior,,"cortical malformation, BPP",,"ear of lynx, ears of grizzly",ears of grizzly,heterotopia,,,present,severe atrophy,severe atrophy,"short CC, hemigenesis",mild atrophy of vermis,,,,ventriculomegaly,AP4B1,"1:113,894,194-113,905,028",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, NN","<1 / 1,000,000",GROWTH\nHeight\n- Short stature\nHEAD & NECK\n...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...","10.1093/brain/awz307 , https://doi.org/10.1002...",# 614066 https://www.omim.org/entry/614066?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,


Would you like to search again?  no


Goodbye!


#### - Testing all unique values for selectability

- Verifies selectability for all unique values given a category

In [35]:
# verify all unique values are selectable for a column
column = 'Cortical Involvement/Subcortical Lesions'
column_values = list(df[column].unique())
unique_values = []
for column_value in column_values:
    values = column_value.split(',')
    for value in values:
        if str(value.strip()) not in unique_values:
            unique_values.append(value.strip())
print(unique_values)

['', 'corona radiata', 'pre and post central gyrus', 'centrum smiovale', 'cortical atrophy', 'corticospinal tracts involvement', 'cortical malformation', 'BPP', 'subcortical lesions', 'global white matter atrophy', 'vascular tortuosity', 'probable microcephaly']


In [58]:
main()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Diffuse or Multifocal



- You chose: [ Diffuse or Multifocal ].




List of Values of Interest:
***************************

D

M



Choose a value of interest:  D



- You chose: [ D ].




Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,...,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
0,SPG1,D,,,,,,,,,...,XLR,NN,"1/300,000 Male",GROWTH\nHeight\n- Short stature (<5-15th perce...,"https://pubmed.ncbi.nlm.nih.gov/8305964/ , 10....",https://doi.org/10.1016/S0387-7604(97)00079-X,# 303350 https://www.omim.org/entry/303350,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,
1,SPG2,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,...,XLR,C,"<1/1,000,000",HEAD & NECK\nEyes\n- Nystagmus\n- Optic atroph...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?sea...,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,Multifocal - MS like lesions
4,SPG5A,"D, M",symmetric,frontal,,"corona radiata, centrum smiovale",,ear of lynx,present,present,...,AR,"C, AD","<1 / 1,000,000",HEAD & NECK:\nEars\n- Sensorineural hearing lo...,"https://doi.org/10.3174/ajnr.A7344 , 10.3389/f...","https://doi.org/10.1016/j.nmd.2008.10.009 , 10...",#270800 https://www.omim.org/entry/270800?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
5,SPG6,D,,,,,,,,,...,AD,"C, AD, A","<1 / 1,000,000",GENITOURINARY\nBladder\n- Urinary urgency\n- U...,"10.3389/fneur.2018.01117 , https://doi.org/10....","https://doi.org/10.1007/s00234-005-1415-3 , ht...",# 600363 https://www.omim.org/entry/600363?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,Only 1 article,
6,SPG7,D,,,,,,,,,...,"AD, AR","C, AD, A, L","1-9/ 100,000",HEAD & NECK\nEyes\n- Optic atrophy\n- Supranuc...,"10.3389/fneur.2018.01117 , https://doi.org/10....",https://doi.org/10.3174/ajnr.A7017,#607259 https://www.omim.org/entry/607259?sear...,https://orpha.net/consor/cgi-bin/Disease_Searc...,,
7,SPG9B,D,symmetric,posterior,,,,present,present,present,...,AR,"I, AD, A","<1 / 1,000,000",GROWTH\nHeight\n- Short stature\nOther\n- Grow...,"10.7860/JCDR/2021/49813.15390 , 10.3389/fneur....",https://doi.org/10.1016/j.braindev.2020.07.015,# 616586 https://www.omim.org/entry/616586?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,Not many articles w/ MRI's. Patient is 2.5 yea...,
8,SPG11,D,symmetric,frontal,,cortical atrophy,,ear of lynx,present,,...,AR,"I, C, AD, A","0.35/100,000",GROWTH\nWeight\n- Obesity\nHEAD & NECK\nEyes\n...,"https://doi.org/10.1002/mds.28519 , https://d...","10.3389/fneur.2018.01117 , https://doi.org/10....",# 604360 https://www.omim.org/entry/604360?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
9,SPG15,D,symmetric,frontal,,,,ear of lynx,present,,...,AR,"C, AD, A","<1 / 1,000,000",HEAD & NECK\nEars\n- Hearing deficit\nEyes\n- ...,"https://doi.org/10.1016/j.ajhg.2008.03.004 , h...","10.3389/fneur.2018.01117 , https://doi.org/10....",# 270700 https://www.omim.org/entry/270700?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
10,SPG17,D,symmetric,,,corticospinal tracts involvement,,,,,...,AD,"C, AD, A","<1 / 1,000,000",SKELETAL\nFeet\n- Pes cavus \nNEUROLOGIC\nCent...,https://www.ncbi.nlm.nih.gov/books/NBK1509/,10.1007/s10072-008-0937-y,# 270685 https://www.omim.org/entry/270685?sea...,,Not many articles w/ MRI's,
11,SPG20,D,symmetric,posterior,,corona radiata,,,mild,present,...,AR,A,"<1 / 1,000,000",GROWTH\nHeight\n- Short stature\nHEAD & NECK\n...,https://www.ncbi.nlm.nih.gov/books/NBK1509/,"https://doi.org/10.1007/s11011-017-0104-3 , h...",# 275900 https://www.omim.org/entry/275900?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,Also called Toyer Syndrome (one of the most co...,


Would you like to search again?  


'Forced exit. Goodbye!'

In [56]:
# verify all unique values are selectable
for col_title in findings_df.columns:
    column_values = list(df[col_title].unique()) # df or findings?
    access_values_of_interest = []
    for column_value in column_values:
        values = column_value.split(',')
        for value in values:
            if str(value.strip()) not in access_values_of_interest:
                access_values_of_interest.append(value.strip())

    # display list of values of interest - Test
    print("List of Values of Interest:"+" ["+col_title+"] *Test")
    print("***************************\n")
    access_values_of_interest.remove('') if ('' in access_values_of_interest) else None
    print('\n\n'.join(access_values_of_interest))
    print()
    
    # display list of values of interest - Version 2
    access_values_of_interest = findings_df[col_title].unique().tolist()
    print("List of Values of Interest:"+" ["+col_title+"] *Version 2")
    print("***************************\n")
    access_values_of_interest.remove('') if ('' in access_values_of_interest) else None
    print('\n\n'.join(access_values_of_interest))
    print()
    print()

List of Values of Interest: [Diffuse or Multifocal] *Test
***************************

D

M

List of Values of Interest: [Diffuse or Multifocal] *Version 2
***************************

D

D, M

M


List of Values of Interest: [Symmetric] *Test
***************************

asymmetric

symmetric

symmetric and asymmetric

List of Values of Interest: [Symmetric] *Version 2
***************************

asymmetric

symmetric

symmetric and asymmetric


List of Values of Interest: [Frontal vs Posterior Predominance] *Test
***************************

posterior

frontal

List of Values of Interest: [Frontal vs Posterior Predominance] *Version 2
***************************

posterior

frontal


List of Values of Interest: [Telltale Grey Matter Involvement] *Test
***************************

T2 hypointense pallidal nuclei

globus pallida calcifications

List of Values of Interest: [Telltale Grey Matter Involvement] *Version 2
***************************

T2 hypointense pallidal nuclei

globus p

#### - General output test 1

* 1st category
- Ventriculomegaly vs Hydrocephalus
- ventriculomegaly

* 2nd category
- Symmetric
- symmetric

* 3rd category
- Periventricular Involvement
- Forceps Major
- present

#### - General output test 2

* 1st category
- Spinal Cord Involvement
- Spinal Cord Atrophy
- present
* 2nd category
- Frontal vs Posterior Predominance
- posterior

In [19]:
main()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Spinal Cord Involvement



- You chose: [ Spinal Cord Involvement ].




List of subcategories:
**********************

Spinal Cord Atrophy

Abnormal Signal/White Matter Tract




Choose a subcategory:  Spinal Cord Atrophy



- You chose: [ Spinal Cord Atrophy ].




List of Values of Interest:
***************************

present

mild

rare



...


- You chose: [ posterior ].




Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
1,SPG2,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial le...",pons,present,,,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,XLR,C,"<1/1,000,000",HEAD & NECK\nEyes\n- Nystagmus\n- Optic atroph...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?sea...,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,Multifocal - MS like lesions
3,SPG4,,symmetric,posterior,,,,,,present,present,,,,,,atrophy of cerebellum,,present,,ventriculomegaly,SPAST,"2:32,063,556-32,157,637+6:35",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"I, C, AD, A","0.9/100,00",HEAD & NECK\nEyes\n- Nystagmus (rare)\nGENITOU...,"10.1186/s12883-022-02595-4 , https://doi.org/...",https://doi.org/10.1007/s00234-005-1415-3; 10....,#182601 https://www.omim.org/entry/182601?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
14,SPG46,D,,posterior,,,,,,present,,,present,present,,,atrophy of vermis,,present,,ventriculomegaly,GBA2,"9:35,736,866-35,749,228",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, C","<1 / 1,000,000",HEAD & NECK\nEars\n- Hearing loss (in some pat...,"10.3389/fneur.2018.01117 , 10.1016/j.ajhg.2012...","https://doi.org/10.1007/s10048-010-0249-2 , 10...",# 614409 https://www.omim.org/entry/614409?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,


Would you like to search again?  no


Goodbye!


In [20]:
main()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Diffuse or Multifocal



- You chose: [ Diffuse or Multifocal ].




List of Values of Interest:
***************************

D

M



Choose a value of interest:  D



- You chose: [ D ].


...


- You chose: [ asymmetric ].




Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
1,SPG2,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial le...",pons,present,,,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,XLR,C,"<1/1,000,000",HEAD & NECK\nEyes\n- Nystagmus\n- Optic atroph...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?sea...,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,Multifocal - MS like lesions
18,SPG50,D,asymmetric,posterior,,"global white matter atrophy, vascular tortuosity",,present,present,present,,,,present,present,short CC,present,,,,asymmetric ventriculomegaly + CSF issue,AP4M1,"7:100,100,794-100,109,039",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, NN","<1 / 1,000,000",HEAD & NECK\nHead\n- Microcephaly \nFace\n- Ps...,"10.3389/fneur.2018.01117 , 10.1002/ajmg.a.3651...","10.1093/brain/awz307 , 10.1212/WNL.00000000000...",# 612936 https://www.omim.org/entry/612936?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,


Would you like to search again?  no


Goodbye!


#### - General output test 3
* Category 1
- Diffuse or Multifocal
- D

* Category 2
- Symmetric
- symmetric

* Category 3
- Periventricular Involvement
- Forceps Major
- ear of lynx

In [32]:
main()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Diffuse or Multifocal



- You chose: [ Diffuse or Multifocal ].




List of Values of Interest:
***************************

D

M



Choose a value of interest:  D



- You chose: [ D ].


...


- You chose: [ ear of lynx ].




Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
4,SPG5A,"D, M",symmetric,frontal,,"corona radiata, centrum smiovale",,ear of lynx,present,present,,present,,present,,,middle cerebellar peduncles,,present,,,CYP7B1,"8:64,586,575-64,798,737",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Both,AR,"C, AD","<1 / 1,000,000",HEAD & NECK:\nEars\n- Sensorineural hearing lo...,"https://doi.org/10.3174/ajnr.A7344 , 10.3389/f...","https://doi.org/10.1016/j.nmd.2008.10.009 , 10...",#270800 https://www.omim.org/entry/270800?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
8,SPG11,D,symmetric,frontal,,cortical atrophy,,ear of lynx,present,,present,present,present,present,,,mild atrophy of vermis,,,,ventriculomegaly,,"15:44,562,696-44,663,662",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, C, AD, A","0.35/100,000",GROWTH\nWeight\n- Obesity\nHEAD & NECK\nEyes\n...,"https://doi.org/10.1002/mds.28519 , https://d...","10.3389/fneur.2018.01117 , https://doi.org/10....",# 604360 https://www.omim.org/entry/604360?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
9,SPG15,D,symmetric,frontal,,,,ear of lynx,present,,,present,present,present,,,mild atrophy of vermis,,rare,,,ZFYVE26,"14:67,728,892-67,816,590",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"C, AD, A","<1 / 1,000,000",HEAD & NECK\nEars\n- Hearing deficit\nEyes\n- ...,"https://doi.org/10.1016/j.ajhg.2008.03.004 , h...","10.3389/fneur.2018.01117 , https://doi.org/10....",# 270700 https://www.omim.org/entry/270700?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
15,SPG47,D,symmetric and asymmetric,posterior,,"cortical malformation, BPP",,"ear of lynx, ears of grizzly",ears of grizzly,heterotopia,,,present,severe atrophy,severe atrophy,"short CC, hemigenesis",mild atrophy of vermis,,,,ventriculomegaly,AP4B1,"1:113,894,194-113,905,028",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, NN","<1 / 1,000,000",GROWTH\nHeight\n- Short stature\nHEAD & NECK\n...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...","10.1093/brain/awz307 , https://doi.org/10.1002...",# 614066 https://www.omim.org/entry/614066?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
16,SPG48,"D, M",symmetric,posterior,,subcortical lesions,,ear of lynx,present,present,,present,present,indenture,,,mild atrophy of cerebellum,,,,,AP5Z1,"7:4,775,623-4,794,397",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AR,A,"<1 / 1,000,000",HEAD & NECK\nEyes\n- Retinopathy (adult onset ...,"https://doi.org/10.1186/s40035-019-0157-9 , ht...","https://doi.org/10.1093/brain/awu121, https://...",# 613647 https://omim.org/entry/613647?search=...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
17,SPG49,D,symmetric,,globus pallida calcifications,,,ear of lynx,present,,,,present,,,,,,,,,TECPR2,,,Complicated,AR,I,"<1 / 1,000,000",,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https:

Would you like to search again?  no


Goodbye!


#### - Test (example 1 from presentation. Result --> SPG2, SPG47 and SPG50)

Diffuse or Multifocal: diffuse

Symmetric: asymmetric

Predominance: posterior

In [2]:
main()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Diffuse or Multifocal



- You chose: [ Diffuse or Multifocal ].




List of Values of Interest:
***************************

D

M



Choose a value of interest:  D



- You chose: [ D ].


...


- You chose: [ posterior ].




Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
1,SPG2,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial le...",pons,present,,,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,XLR,C,"<1/1,000,000",HEAD & NECK\nEyes\n- Nystagmus\n- Optic atroph...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?sea...,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,Multifocal - MS like lesions
15,SPG47,D,"symmetric, asymmetric",posterior,,"cortical malformation, BPP",,"ear of lynx, ears of grizzly",ears of grizzly,heterotopia,,,present,severe atrophy,severe atrophy,"short CC, hemigenesis",mild atrophy of vermis,,,,ventriculomegaly,AP4B1,"1:113,894,194-113,905,028",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, NN","<1 / 1,000,000",GROWTH\nHeight\n- Short stature\nHEAD & NECK\n...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...","10.1093/brain/awz307 , https://doi.org/10.1002...",# 614066 https://www.omim.org/entry/614066?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
18,SPG50,D,asymmetric,posterior,,"global white matter atrophy, vascular tortuosity",,present,present,present,,,,present,present,short CC,present,,,,asymmetric ventriculomegaly + CSF issue,AP4M1,"7:100,100,794-100,109,039",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, NN","<1 / 1,000,000",HEAD & NECK\nHead\n- Microcephaly \nFace\n- Ps...,"10.3389/fneur.2018.01117 , 10.1002/ajmg.a.3651...","10.1093/brain/awz307 , 10.1212/WNL.00000000000...",# 612936 https://www.omim.org/entry/612936?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,


Would you like to search again?  no


Goodbye!


#### - Test: Order of rows dependings on results (with example 1)

In [75]:
main()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Diffuse or Multifocal



- You chose: [ Diffuse or Multifocal ].




List of Values of Interest:
***************************

D

M



Choose a value of interest:  D



- You chose: [ D ].


...


- You chose: [ posterior ].




Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
1,SPG2,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial le...",pons,present,,,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,XLR,C,"<1/1,000,000",HEAD & NECK\nEyes\n- Nystagmus\n- Optic atroph...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?sea...,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,Multifocal - MS like lesions
15,SPG47,D,"symmetric, asymmetric",posterior,,"cortical malformation, BPP",,"ear of lynx, ears of grizzly",ears of grizzly,heterotopia,,,present,severe atrophy,severe atrophy,"short CC, hemigenesis",mild atrophy of vermis,,,,ventriculomegaly,AP4B1,"1:113,894,194-113,905,028",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, NN","<1 / 1,000,000",GROWTH\nHeight\n- Short stature\nHEAD & NECK\n...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...","10.1093/brain/awz307 , https://doi.org/10.1002...",# 614066 https://www.omim.org/entry/614066?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
18,SPG50,D,asymmetric,posterior,,"global white matter atrophy, vascular tortuosity",,present,present,present,,,,present,present,short CC,present,,,,asymmetric ventriculomegaly + CSF issue,AP4M1,"7:100,100,794-100,109,039",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, NN","<1 / 1,000,000",HEAD & NECK\nHead\n- Microcephaly \nFace\n- Ps...,"10.3389/fneur.2018.01117 , 10.1002/ajmg.a.3651...","10.1093/brain/awz307 , 10.1212/WNL.00000000000...",# 612936 https://www.omim.org/entry/612936?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
0,SPG1,D,,,,,,,,,agenesis,agenesis,,,,,,,,,hydrocephalus,L1CAM,"X:153,861,516-153,886,173",https://www.ncbi.nlm.nih.gov/gene/3897,Complicated,XLR,NN,"1/300,000 Male",GROWTH\nHeight\n- Short stature (<5-15th perce...,"https://pubmed.ncbi.nlm.nih.gov/8305964/ , 10....",https://doi.org/10.1016/S0387-7604(97)00079-X,# 303350 https://www.omim.org/entry/303350,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,
2,SPG3A,M,,posterior,,"corona radiata, pre central gyrus, post centra...",,,,present,,,,,,,,,mild,,,ATL1,"14:50,533,082-50,633,068",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"C, A","1-9 / 1,000,000",GENITOURINARY\nBladder\n- Urinary urgency\n- U...,"10.3389/fneur.2023.1239725 , https://www.ncbi....","https://doi.org/10.1007/s00234-005-1415-3 , ht...",#182600 https://www.omim.org/entry/182600?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
3,SPG4,,symmetric,posterior,,,,,,present,present,,,,,,atrophy of cerebellum,,present,,ventriculomegaly,SPAST,"2:32,063,556-32,157,637+6:35",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"I, C, AD, A","0.9/100,00",HEAD & NECK\nEyes\n- Nystagmus (rare)\nGE

Would you like to search again?  no


Goodbye!


## Main

In [48]:
def main():
    
    # imports
    import pandas as pd # data manipulation/access
    import utils # custom utility functions
    from IPython.display import display, HTML # hyperlink access
    #import warnings
    #warnings.simplefilter(action='ignore', category=FutureWarning)
    pd.set_option('display.max_columns', None)
    #import fuzzywuzzy # fuzzy match
    #pd.set_option('display.max_colwidth', None)
    #import re # regex

    dimension_1 = ['Diffuse or Multifocal','Symmetric','Frontal vs Posterior Predominance','Telltale Grey Matter Involvement','Cortical Involvement/Subcortical Lesions','U-Fiber/Juxtacortical Involvement', 'Ventriculomegaly vs Hydrocephalus']
    dimension_2 = ['Spinal Cord Involvement','Periventricular Involvement']
    dimension_3 = ['Subcortical Structures']

    # main loop
    flag = True
    while flag:

        # data
        findings_df = pd.read_excel('hsp_data.xlsx').iloc[:, list(range(0,20))].fillna('')
        df = pd.read_excel('hsp_data.xlsx').fillna('')     

        # initially, additional_category is set to True to start a search
        # if set to False, return results with the only selected search category
        additional_category_flag = True

        # keep track of all choices made by the user
        all_choices = []
        while additional_category_flag:
            
            # display list of main categories
            access_categories = list(utils.findings_categorized.keys())
            print("List of categories:")
            print("*******************\n")
            print('\n\n'.join(access_categories))
            print()
            print()

            # user input for main category #
            category = input("Enter name of MRI finding: ")
            
            # keyboard interrupt
            if category == "":
                return "Forced exit."
            
            # invalid input handling #
            while category not in access_categories:
                print()
                category = input("Invalid input. Choose a category from the list above (case sensitive): ")
            
            print()
            print("- You chose: [", category,"].")
            print()
            print()
            
            # check if main category is more than 1 dimension
            if category not in dimension_1:

                # 3 dimension
                if category not in dimension_2:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized['Subcortical Structures']
                    print("List of subcategories:")
                    print("**********************\n")
                    print('\n\n'.join(list(access_subcategories)))
                    print()
                    print()

                    # user input for subcategory #
                    subcategory = input("Choose a subcategory: ")

                    # keyboard interrupt
                    if subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while subcategory not in access_subcategories:
                        print()
                        subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")

                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # display list of sub-subcategories
                    access_sub_subcategories = access_subcategories[subcategory]
                    print("List of sub_subcategories:")
                    print("**************************\n")
                    print('\n\n'.join(list(access_sub_subcategories)))
                    print()
                    print()

                    # user input for sub_subcategory #
                    sub_subcategory = input("Choose a sub-subcategory: ")

                    # keyboard interrupt
                    if sub_subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while sub_subcategory not in access_sub_subcategories:
                        print()
                        sub_subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")
                        
                    print()
                    print("- You chose: [", sub_subcategory,"].")

                    # column title
                    col_title = str(category+':'+subcategory+':'+sub_subcategory)

                # 2 dimension
                else:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized[category]
                    print("\n\nList of subcategories:")
                    print("**********************\n")
                    print('\n\n'.join(list(access_subcategories)))
                    print()
                    print()

                    # user input for subcategory #
                    subcategory = input("Choose a subcategory: ")

                    # keyboard interrupt
                    if subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while subcategory not in access_subcategories:
                        print()
                        subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")

                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # column title
                    col_title = str(category+':'+subcategory)

            # 1 dimension        
            else:

                # column title
                col_title = category
            print()
            print()

            # verify all unique values are selectable
            column_values = list(df[col_title].unique())
            access_values_of_interest = []
            for column_value in column_values:
                values = column_value.split(',')
                for value in values:
                    if str(value.strip()) not in access_values_of_interest:
                        access_values_of_interest.append(value.strip())
                        
            # display list of values of interest
            print("List of Values of Interest:")
            print("***************************\n")
            access_values_of_interest.remove('') if ('' in access_values_of_interest) else None
            print('\n\n'.join(access_values_of_interest))
            print()

            # user input for value of interest #
            value_of_interest = input("Choose a value of interest: ")

            # keyboard interrupt
            if value_of_interest == "":
                return "Forced exit."
            
            # invalid input handling #
            while value_of_interest not in access_values_of_interest:
                print()
                value_of_interest = input("Invalid input. Choose a value of interest from the list above (case sensitive): ")

            print()
            print("- You chose: [", value_of_interest,"].")

            # keep relevant rows
            findings_df = findings_df[findings_df[col_title].str.contains(pat=r'\b{}\b'.format(value_of_interest), regex=True)]

            # update list of user choices #############################################################
            all_choices.append(str(col_title+"%"+value_of_interest))
            
            # prompt user to add additional search constraint #
            add_a_category = input("\n\nWould you to add another category? ")

            # keyboard interrupt
            if add_a_category == "":
                return "Forced exit."

            if add_a_category == '0' or add_a_category.lower()=="no":
                additional_category_flag = False
        print()
        print()
        
        # move "Gene" to first column
        current_columns = df.columns.tolist()
        df = df[['Gene'] + [col for col in current_columns if col != 'Gene']]
        
        # highlight relevant rows
        mask = df.index.isin(findings_df.index.tolist())
        result_rows = findings_df.index.tolist()
        def highlight_rows(row):
            if row.name in result_rows:
                return ['background-color: steelblue'] * len(row)
            else:
                return [''] * len(row)

        # display main results
        result_to_display = pd.DataFrame(df.loc[result_rows]).drop(["Clinical Synopsis"], axis=1)
        results = result_to_display.style.apply(highlight_rows, axis=1)
        display(results)
        print()

        # display rest of df ########################
        rest_of_data = df[~mask]
        rest_of_data["Rank"] = 0
        for i in range(len(all_choices)):
            col = all_choices[i].split('%')[0]
            value = all_choices[i].split('%')[1]
            rest_of_data.loc[rest_of_data[col]==value, 'Rank'] += 1
        rest_of_data = pd.DataFrame(rest_of_data.sort_values(by='Rank', ascending=False))
        display(pd.DataFrame(rest_of_data))

        # prompt user to start new search #
        #print(all_choices)
        terminate = input("Would you like to search again? ")
        if terminate == '0' or terminate.lower() == 'no' or terminate == "":
            flag = False

    # exit message
    print("Goodbye!")

### Example 1: (3)
Diffuse or Multifocal: diffuse

Symmetric: asymmetric

Predominance: posterior

In [4]:
main()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Diffuse or Multifocal



- You chose: [ Diffuse or Multifocal ].




List of Values of Interest:
***************************

D

M



Choose a value of interest:  D



- You chose: [ D ].


...



Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
1,SPG2,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial lesions",pons,present,,,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chrX:103776506-103792619&dgv=pack&knownGene=pack&omimGene=pack,Complicated,XLR,C,"<1/1,000,000","10.3389/fneur.2018.01117 , https://www.ncbi.nlm.nih.gov/books/NBK1509/",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?search=spg2&highlight=spg2,"https://www.orpha.net/consor/cgi-bin/OC_Exp.php?lng=EN&Expert=99015#:~:text=Pure%20SPG2%20manifests%20as%20early,gait%20due%20to%20spastic%20paraparesis.",,Multifocal - MS like lesions
15,SPG47,D,"symmetric, asymmetric",posterior,,"cortical malformation, BPP",,"ear of lynx, ears of grizzly",ears of grizzly,heterotopia,,,present,severe atrophy,severe atrophy,"short CC, hemigenesis",mild atrophy of vermis,,,,ventriculomegaly,AP4B1,"1:113,894,194-113,905,028",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chr1:113894194-113905028&dgv=pack&knownGene=pack&omimGene=pack,Complicated,AR,"I, NN","<1 / 1,000,000","10.3389/fneur.2018.01117 , https://www.ncbi.nlm.nih.gov/books/NBK1509/","10.1093/brain/awz307 , https://doi.org/10.1002/ajmg.a.38561 , 10.1212/WNL.0000000000012836 , https://doi.org/10.1016/j.jns.2011.03.011 , https://doi-org.proxy3.library.mcgill.ca/10.1002/ajmg.a.36514",# 614066 https://www.omim.org/entry/614066?search=spg47&highlight=spg47,https://www.orpha.net/consor/cgi-bin/Disease_Search.php?lng=EN&data_id=20498&Disease_Disease_Search_diseaseGroup=spg47&Disease_Disease_Search_diseaseType=Pat&Disease(s)/group%20of%20diseases=Severe-intellectual-disability-and-progressive-spastic-paraplegia&title=Severe%20intellectual%20disability%20and%20progressive%20spastic%20paraplegia&search=Disease_Search_Simple,,
18,SPG50,D,asymmetric,posterior,,"global white matter atrophy, vascular tortuosity",,present,present,present,,,,present,present,short CC,present,,,,asymmetric ventriculomegaly and CSF issue,AP4M1,"7:100,100,794-100,109,039",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chr7:100100794-100109039&dgv=pack&knownGene=pack&omimGene=pack,Complicated,AR,"I, NN","<1 / 1,000,000","10.3389/fneur.2018.01117 , 10.1002/ajmg.a.36514 , https://www.ncbi.nlm.nih.gov/books/NBK1509/","10.1093/brain/awz307 , 10.1212/WNL.0000000000012836 , https://doi-org.proxy3.library.mcgill.ca/10.1002/ajmg.a.36514 , 10.1016/j.ajhg.2009.06.004 , https://doi.org/10.1212%2FNXG.0000000000000217",# 612936 https://www.omim.org/entry/612936?search=spg50&highlight=spg50,https://www.orpha.net/consor/cgi-bin/Disease_Search.php?lng=EN&data_id=20498&Disease_Disease_Search_diseaseGroup=spg50&Disease_Disease_Search_diseaseType=Pat&Disease(s)/group%20of%20diseases=Severe-intellectual-disability-and-progressive-spastic-paraplegia&title=Severe%20intellec

,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
0,SPG1,D,,,,,,,,,agenesis,agenesis,,,,,,,,,hydrocephalus,L1CAM,"X:153,861,516-153,886,173",https://www.ncbi.nlm.nih.gov/gene/3897,Complicated,XLR,NN,"1/300,000 Male",GROWTH\nHeight\n- Short stature (<5-15th perce...,"https://pubmed.ncbi.nlm.nih.gov/8305964/ , 10....",https://doi.org/10.1016/S0387-7604(97)00079-X,# 303350 https://www.omim.org/entry/303350,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,
2,SPG3A,M,,posterior,,"corona radiata, pre central gyrus, post centra...",,,,present,,,,,,,,,mild,,,ATL1,"14:50,533,082-50,633,068",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"C, A","1-9 / 1,000,000",GENITOURINARY\nBladder\n- Urinary urgency\n- U...,"10.3389/fneur.2023.1239725 , https://www.ncbi....","https://doi.org/10.1007/s00234-005-1415-3 , ht...",#182600 https://www.omim.org/entry/182600?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
3,SPG4,,symmetric,posterior,,,,,,present,present,,,,,,atrophy of cerebellum,,present,,ventriculomegaly,SPAST,"2:32,063,556-32,157,637+6:35",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"I, C, AD, A","0.9/100,00",HEAD & NECK\nEyes\n- Nystagmus (rare)\nGENITOU...,"10.1186/s12883-022-02595-4 , https://doi.org/...",https://doi.org/10.1007/s00234-005-1415-3; 10....,#182601 https://www.omim.org/entry/182601?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
4,SPG5A,"D, M",symmetric,frontal,,"corona radiata, centrum smiovale",,ear of lynx,present,present,,present,,present,,,middle cerebellar peduncles,,present,,,CYP7B1,"8:64,586,575-64,798,737",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Both,AR,"C, AD","<1 / 1,000,000",HEAD & NECK\nEars\n- Sensorineural hearing los...,"https://doi.org/10.3174/ajnr.A7344 , 10.3389/f...","https://doi.org/10.1016/j.nmd.2008.10.009 , 10...",#270800 https://www.omim.org/entry/270800?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
5,SPG6,D,,,,,,,,,,,,,,,,,present,,,NIPA1,"15:22,786,225-22,829,789",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"C, AD, A","<1 / 1,000,000",GENITOURINARY\n\nBladder\n- Urinary urgency\n-...,"10.3389/fneur.2018.01117 , https://doi.org/10....","https://doi.org/10.1007/s00234-005-1415-3 , ht...",# 600363 https://www.omim.org/entry/600363?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,Only 1 article,
6,SPG7,D,,,,,,,,,,,mild,,,,atrophy of cerebellum,,,,,PGN,"16:89,508,388-89,557,768",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Both,"AD, AR","C, AD, A, L","1-9/ 100,000",HEAD & NECK\nEyes\n- Optic atrophy\n- Supranuc...,"10.3389/fneur.2018.01117 , https://doi.org/10....",https://doi.org/10.3174/ajnr.A7017,#607259 https://www.omim.org/entry/607259?sear...,https://orpha.net/consor/cgi-bin/Disease_Searc...,,
7,SPG9B,D,symmetric,posterior,,,,present,present,present,,,,,present,,mild atrophy of vermis,,,,ventriculomegaly,ALDH18A1,"10:95,605,941-95,656,7

Would you like to search again?  


Goodbye!


### Example 2: (2) 

Diffuse or Multifocal: diffuse

Symmetric: symmetric

Cortical involvement: cortical atrophy

Periventricular: forceps minor, present

Subcortical: corpus callosum: genu, body, isthmus

Subcortical: posterior fossa: cerebellar, mild atrophy of vermis

### Example 3: (2)

Diffuse or Multifocal: diffuse and multifocal

Symmetric: symmetric

Periventricular: forceps major: ear of lynx

Periventricular: forceps minor: present

Periventricular: other: present

In [ ]:
main()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Diffuse or Multifocal



- You chose: [ Diffuse or Multifocal ].




List of Values of Interest:
***************************

D

M



Choose a value of interest:  D



- You chose: [ D ].


...

Choose a value of interest:  present



- You chose: [ present ].




Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
4,SPG5A,"D, M",symmetric,frontal,,"corona radiata, centrum smiovale",,ear of lynx,present,present,,present,,present,,,middle cerebellar peduncles,,present,,,CYP7B1,"8:64,586,575-64,798,737",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chr8:64586575-64798737&dgv=pack&knownGene=pack&omimGene=pack,Both,AR,"C, AD","<1 / 1,000,000","https://doi.org/10.3174/ajnr.A7344 , 10.3389/fneur.2023.1239725 , https://doi.org/10.1186/s40035-019-0157-9 , https://www.ncbi.nlm.nih.gov/books/NBK1509/","https://doi.org/10.1016/j.nmd.2008.10.009 , 10.1093/brain/awp073",#270800 https://www.omim.org/entry/270800?search=spg5&highlight=spg5,https://www.orpha.net/consor/cgi-bin/Disease_Search.php?lng=EN&data_id=14697&Disease_Disease_Search_diseaseGroup=spg5a&Disease_Disease_Search_diseaseType=Pat&Disease(s)/group%20of%20diseases=Autosomal-recessive-spastic-paraplegia-type-5A&title=Autosomal%20recessive%20spastic%20paraplegia%20type%205A&search=Disease_Search_Simple,,
16,SPG48,"D, M",symmetric,posterior,,subcortical lesions,,ear of lynx,present,present,,present,present,indenture,,,mild atrophy of cerebellum,,,,,AP5Z1,"7:4,775,623-4,794,397",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chr7:4775623-4794397&dgv=pack&knownGene=pack&omimGene=pack,Pure,AR,A,"<1 / 1,000,000","https://doi.org/10.1186/s40035-019-0157-9 , https://www.ncbi.nlm.nih.gov/books/NBK1509/","https://doi.org/10.1093/brain/awu121, https://doi.org/10.3389/fneur.2023.1156100,",# 613647 https://omim.org/entry/613647?search=spg48&highlight=spg48,https://www.orpha.net/consor/cgi-bin/Disease_Search.php?lng=EN&data_id=21221&Disease_Disease_Search_diseaseGroup=spg48&Disease_Disease_Search_diseaseType=Pat&Disease(s)/group%20of%20diseases=Autosomal-recessive-spastic-paraplegia-type-48&title=Autosomal%20recessive%20spastic%20paraplegia%20type%2048&search=Disease_Search_Simple,,


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
0,SPG1,D,,,,,,,,,agenesis,agenesis,,,,,,,,,hydrocephalus,L1CAM,"X:153,861,516-153,886,173",https://www.ncbi.nlm.nih.gov/gene/3897,Complicated,XLR,NN,"1/300,000 Male",GROWTH\nHeight\n- Short stature (<5-15th perce...,"https://pubmed.ncbi.nlm.nih.gov/8305964/ , 10....",https://doi.org/10.1016/S0387-7604(97)00079-X,# 303350 https://www.omim.org/entry/303350,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,
1,SPG2,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial le...",pons,present,,,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,XLR,C,"<1/1,000,000",HEAD & NECK\nEyes\n- Nystagmus\n- Optic atroph...,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?sea...,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,Multifocal - MS like lesions
2,SPG3A,M,,posterior,,"corona radiata, pre central gyrus, post centra...",,,,present,,,,,,,,,mild,,,ATL1,"14:50,533,082-50,633,068",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"C, A","1-9 / 1,000,000",GENITOURINARY\nBladder\n- Urinary urgency\n- U...,"10.3389/fneur.2023.1239725 , https://www.ncbi....","https://doi.org/10.1007/s00234-005-1415-3 , ht...",#182600 https://www.omim.org/entry/182600?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
3,SPG4,,symmetric,posterior,,,,,,present,present,,,,,,atrophy of cerebellum,,present,,ventriculomegaly,SPAST,"2:32,063,556-32,157,637+6:35",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"I, C, AD, A","0.9/100,00",HEAD & NECK\nEyes\n- Nystagmus (rare)\nGENITOU...,"10.1186/s12883-022-02595-4 , https://doi.org/...",https://doi.org/10.1007/s00234-005-1415-3; 10....,#182601 https://www.omim.org/entry/182601?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
5,SPG6,D,,,,,,,,,,,,,,,,,present,,,NIPA1,"15:22,786,225-22,829,789",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"C, AD, A","<1 / 1,000,000",GENITOURINARY\n\nBladder\n- Urinary urgency\n-...,"10.3389/fneur.2018.01117 , https://doi.org/10....","https://doi.org/10.1007/s00234-005-1415-3 , ht...",# 600363 https://www.omim.org/entry/600363?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,Only 1 article,
6,SPG7,D,,,,,,,,,,,mild,,,,atrophy of cerebellum,,,,,PGN,"16:89,508,388-89,557,768",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Both,"AD, AR","C, AD, A, L","1-9/ 100,000",HEAD & NECK\nEyes\n- Optic atrophy\n- Supranuc...,"10.3389/fneur.2018.01117 , https://doi.org/10....",https://doi.org/10.3174/ajnr.A7017,#607259 https://www.omim.org/entry/607259?sear...,https://orpha.net/consor/cgi-bin/Disease_Searc...,,
7,SPG9B,D,symmetric,posterior,,,,present,present,present,,,,,present,,mild atrophy of vermis,,,,ventriculomegaly,ALDH18A1,"10:95,605,941-9

In [5]:
main()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Diffuse or Multifocal



- You chose: [ Diffuse or Multifocal ].




List of Values of Interest:
***************************

D

M



Choose a value of interest:  D



- You chose: [ D ].


...



Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
1,SPG2,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial lesions",pons,present,,,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chrX:103776506-103792619&dgv=pack&knownGene=pack&omimGene=pack,Complicated,XLR,C,"<1/1,000,000","10.3389/fneur.2018.01117 , https://www.ncbi.nlm.nih.gov/books/NBK1509/",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?search=spg2&highlight=spg2,"https://www.orpha.net/consor/cgi-bin/OC_Exp.php?lng=EN&Expert=99015#:~:text=Pure%20SPG2%20manifests%20as%20early,gait%20due%20to%20spastic%20paraparesis.",,Multifocal - MS like lesions
15,SPG47,D,"symmetric, asymmetric",posterior,,"cortical malformation, BPP",,"ear of lynx, ears of grizzly",ears of grizzly,heterotopia,,,present,severe atrophy,severe atrophy,"short CC, hemigenesis",mild atrophy of vermis,,,,ventriculomegaly,AP4B1,"1:113,894,194-113,905,028",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chr1:113894194-113905028&dgv=pack&knownGene=pack&omimGene=pack,Complicated,AR,"I, NN","<1 / 1,000,000","10.3389/fneur.2018.01117 , https://www.ncbi.nlm.nih.gov/books/NBK1509/","10.1093/brain/awz307 , https://doi.org/10.1002/ajmg.a.38561 , 10.1212/WNL.0000000000012836 , https://doi.org/10.1016/j.jns.2011.03.011 , https://doi-org.proxy3.library.mcgill.ca/10.1002/ajmg.a.36514",# 614066 https://www.omim.org/entry/614066?search=spg47&highlight=spg47,https://www.orpha.net/consor/cgi-bin/Disease_Search.php?lng=EN&data_id=20498&Disease_Disease_Search_diseaseGroup=spg47&Disease_Disease_Search_diseaseType=Pat&Disease(s)/group%20of%20diseases=Severe-intellectual-disability-and-progressive-spastic-paraplegia&title=Severe%20intellectual%20disability%20and%20progressive%20spastic%20paraplegia&search=Disease_Search_Simple,,
18,SPG50,D,asymmetric,posterior,,"global white matter atrophy, vascular tortuosity",,present,present,present,,,,present,present,short CC,present,,,,asymmetric ventriculomegaly and CSF issue,AP4M1,"7:100,100,794-100,109,039",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chr7:100100794-100109039&dgv=pack&knownGene=pack&omimGene=pack,Complicated,AR,"I, NN","<1 / 1,000,000","10.3389/fneur.2018.01117 , 10.1002/ajmg.a.36514 , https://www.ncbi.nlm.nih.gov/books/NBK1509/","10.1093/brain/awz307 , 10.1212/WNL.0000000000012836 , https://doi-org.proxy3.library.mcgill.ca/10.1002/ajmg.a.36514 , 10.1016/j.ajhg.2009.06.004 , https://doi.org/10.1212%2FNXG.0000000000000217",# 612936 https://www.omim.org/entry/612936?search=spg50&highlight=spg50,https://www.orpha.net/consor/cgi-bin/Disease_Search.php?lng=EN&data_id=20498&Disease_Disease_Search_diseaseGroup=spg50&Disease_Disease_Search_diseaseType=Pat&Disease(s)/group%20of%20diseases=Severe-intellectual-disability-and-progressive-spastic-paraplegia&title=Severe%20intellec

,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
0,SPG1,D,,,,,,,,,agenesis,agenesis,,,,,,,,,hydrocephalus,L1CAM,"X:153,861,516-153,886,173",https://www.ncbi.nlm.nih.gov/gene/3897,Complicated,XLR,NN,"1/300,000 Male",GROWTH\nHeight\n- Short stature (<5-15th perce...,"https://pubmed.ncbi.nlm.nih.gov/8305964/ , 10....",https://doi.org/10.1016/S0387-7604(97)00079-X,# 303350 https://www.omim.org/entry/303350,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,
2,SPG3A,M,,posterior,,"corona radiata, pre central gyrus, post centra...",,,,present,,,,,,,,,mild,,,ATL1,"14:50,533,082-50,633,068",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"C, A","1-9 / 1,000,000",GENITOURINARY\nBladder\n- Urinary urgency\n- U...,"10.3389/fneur.2023.1239725 , https://www.ncbi....","https://doi.org/10.1007/s00234-005-1415-3 , ht...",#182600 https://www.omim.org/entry/182600?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
3,SPG4,,symmetric,posterior,,,,,,present,present,,,,,,atrophy of cerebellum,,present,,ventriculomegaly,SPAST,"2:32,063,556-32,157,637+6:35",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"I, C, AD, A","0.9/100,00",HEAD & NECK\nEyes\n- Nystagmus (rare)\nGENITOU...,"10.1186/s12883-022-02595-4 , https://doi.org/...",https://doi.org/10.1007/s00234-005-1415-3; 10....,#182601 https://www.omim.org/entry/182601?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
4,SPG5A,"D, M",symmetric,frontal,,"corona radiata, centrum smiovale",,ear of lynx,present,present,,present,,present,,,middle cerebellar peduncles,,present,,,CYP7B1,"8:64,586,575-64,798,737",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Both,AR,"C, AD","<1 / 1,000,000",HEAD & NECK\nEars\n- Sensorineural hearing los...,"https://doi.org/10.3174/ajnr.A7344 , 10.3389/f...","https://doi.org/10.1016/j.nmd.2008.10.009 , 10...",#270800 https://www.omim.org/entry/270800?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
5,SPG6,D,,,,,,,,,,,,,,,,,present,,,NIPA1,"15:22,786,225-22,829,789",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"C, AD, A","<1 / 1,000,000",GENITOURINARY\n\nBladder\n- Urinary urgency\n-...,"10.3389/fneur.2018.01117 , https://doi.org/10....","https://doi.org/10.1007/s00234-005-1415-3 , ht...",# 600363 https://www.omim.org/entry/600363?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,Only 1 article,
6,SPG7,D,,,,,,,,,,,mild,,,,atrophy of cerebellum,,,,,PGN,"16:89,508,388-89,557,768",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Both,"AD, AR","C, AD, A, L","1-9/ 100,000",HEAD & NECK\nEyes\n- Optic atrophy\n- Supranuc...,"10.3389/fneur.2018.01117 , https://doi.org/10....",https://doi.org/10.3174/ajnr.A7017,#607259 https://www.omim.org/entry/607259?sear...,https://orpha.net/consor/cgi-bin/Disease_Searc...,,
7,SPG9B,D,symmetric,posterior,,,,present,present,present,,,,,present,,mild atrophy of vermis,,,,ventriculomegaly,ALDH18A1,"10:95,605,941-95,656,7

Would you like to search again?  


Goodbye!


## Choice Priority

In [10]:
# data
df = pd.read_excel('hsp_data.xlsx').fillna('')
findings_df = pd.read_excel('hsp_data.xlsx').iloc[:, list(range(0,20))].fillna('')

# save choices
choices_list = []
temp = []

value_of_interest = 'D'
category = 'Diffuse or Multifocal'

temp.append([category])
temp[0].append(value_of_interest)

# show choices


TypeError: 'dict' object is not callable

In [4]:
# patient 1 (sister)
test()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Diffuse or Multifocal



- You chose: [ Diffuse or Multifocal ].




List of Values of Interest:
***************************

D

M



Choose a value of interest:  D



- You chose: [ D ].


...



Would you to add another category?  no


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
1,SPG2,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial lesions",pons,present,,,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chrX:103776506-103792619&dgv=pack&knownGene=pack&omimGene=pack,Complicated,XLR,C,"<1/1,000,000","10.3389/fneur.2018.01117 , https://www.ncbi.nlm.nih.gov/books/NBK1509/",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?search=spg2&highlight=spg2,"https://www.orpha.net/consor/cgi-bin/OC_Exp.php?lng=EN&Expert=99015#:~:text=Pure%20SPG2%20manifests%20as%20early,gait%20due%20to%20spastic%20paraparesis.",,Multifocal - MS like lesions
4,SPG5A,"D, M",symmetric,frontal,,"corona radiata, centrum smiovale",,ear of lynx,present,present,,present,,present,,,middle cerebellar peduncles,,present,,,CYP7B1,"8:64,586,575-64,798,737",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chr8:64586575-64798737&dgv=pack&knownGene=pack&omimGene=pack,Both,AR,"C, AD","<1 / 1,000,000","https://doi.org/10.3174/ajnr.A7344 , 10.3389/fneur.2023.1239725 , https://doi.org/10.1186/s40035-019-0157-9 , https://www.ncbi.nlm.nih.gov/books/NBK1509/","https://doi.org/10.1016/j.nmd.2008.10.009 , 10.1093/brain/awp073",#270800 https://www.omim.org/entry/270800?search=spg5&highlight=spg5,https://www.orpha.net/consor/cgi-bin/Disease_Search.php?lng=EN&data_id=14697&Disease_Disease_Search_diseaseGroup=spg5a&Disease_Disease_Search_diseaseType=Pat&Disease(s)/group%20of%20diseases=Autosomal-recessive-spastic-paraplegia-type-5A&title=Autosomal%20recessive%20spastic%20paraplegia%20type%205A&search=Disease_Search_Simple,,
7,SPG9B,D,symmetric,posterior,,,,present,present,present,,,,,present,,mild atrophy of vermis,,,,ventriculomegaly,ALDH18A1,"10:95,605,941-95,656,711",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chr10:95605941-95656711&dgv=pack&knownGene=pack&omimGene=pack,Complicated,AR,"I, AD, A","<1 / 1,000,000","10.7860/JCDR/2021/49813.15390 , 10.3389/fneur.2023.1239725 , https://doi.org/10.1186/s40035-019-0157-9 , https://www.ncbi.nlm.nih.gov/books/NBK1509/",https://doi.org/10.1016/j.braindev.2020.07.015,# 616586 https://www.omim.org/entry/616586?search=spg9b&highlight=spg9b,https://www.orpha.net/consor/cgi-bin/Disease_Search.php?lng=EN&data_id=23538&Disease_Disease_Search_diseaseGroup=spg9b&Disease_Disease_Search_diseaseType=Pat&Disease(s)/group%20of%20diseases=Autosomal-recessive-spastic-paraplegia-type-9B&title=Autosomal%20recessive%20spastic%20paraplegia%20type%209B&search=Disease_Search_Simple,Not many articles w/ MRI's. Patient is 2.5 years old so hypomyelination questionable.,
12,SPG21,D,symmetric,posterior,,cortical atrophy,,,present,present,,present,present,present,,,mild atrophy of vermis,,,,,ACP33,"15:64,963,022-64,989,914",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chr15:6

,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
0,SPG1,D,,,,,,,,,agenesis,agenesis,,,,,,,,,hydrocephalus,L1CAM,"X:153,861,516-153,886,173",https://www.ncbi.nlm.nih.gov/gene/3897,Complicated,XLR,NN,"1/300,000 Male",GROWTH\nHeight\n- Short stature (<5-15th perce...,"https://pubmed.ncbi.nlm.nih.gov/8305964/ , 10....",https://doi.org/10.1016/S0387-7604(97)00079-X,# 303350 https://www.omim.org/entry/303350,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,
2,SPG3A,M,,posterior,,"corona radiata, pre central gyrus, post centra...",,,,present,,,,,,,,,mild,,,ATL1,"14:50,533,082-50,633,068",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"C, A","1-9 / 1,000,000",GENITOURINARY\nBladder\n- Urinary urgency\n- U...,"10.3389/fneur.2023.1239725 , https://www.ncbi....","https://doi.org/10.1007/s00234-005-1415-3 , ht...",#182600 https://www.omim.org/entry/182600?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
3,SPG4,,symmetric,posterior,,,,,,present,present,,,,,,atrophy of cerebellum,,present,,ventriculomegaly,SPAST,"2:32,063,556-32,157,637+6:35",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"I, C, AD, A","0.9/100,00",HEAD & NECK\nEyes\n- Nystagmus (rare)\nGENITOU...,"10.1186/s12883-022-02595-4 , https://doi.org/...",https://doi.org/10.1007/s00234-005-1415-3; 10....,#182601 https://www.omim.org/entry/182601?sear...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
5,SPG6,D,,,,,,,,,,,,,,,,,present,,,NIPA1,"15:22,786,225-22,829,789",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Pure,AD,"C, AD, A","<1 / 1,000,000",GENITOURINARY\n\nBladder\n- Urinary urgency\n-...,"10.3389/fneur.2018.01117 , https://doi.org/10....","https://doi.org/10.1007/s00234-005-1415-3 , ht...",# 600363 https://www.omim.org/entry/600363?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,Only 1 article,
6,SPG7,D,,,,,,,,,,,mild,,,,atrophy of cerebellum,,,,,PGN,"16:89,508,388-89,557,768",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Both,"AD, AR","C, AD, A, L","1-9/ 100,000",HEAD & NECK\nEyes\n- Optic atrophy\n- Supranuc...,"10.3389/fneur.2018.01117 , https://doi.org/10....",https://doi.org/10.3174/ajnr.A7017,#607259 https://www.omim.org/entry/607259?sear...,https://orpha.net/consor/cgi-bin/Disease_Searc...,,
8,SPG11,D,symmetric,frontal,,cortical atrophy,,ear of lynx,present,,present,present,present,present,,,mild atrophy of vermis,,,,ventriculomegaly,,"15:44,562,696-44,663,662",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, C, AD, A","0.35/100,000",GROWTH\n\nWeight\n- Obesity\nHEAD & NECK\nEyes...,"https://doi.org/10.1002/mds.28519 , https://d...","10.3389/fneur.2018.01117 , https://doi.org/10....",# 604360 https://www.omim.org/entry/604360?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,
9,SPG15,D,symmetric,frontal,,,,ear of lynx,present,,,present,present,present,,,mild atrophy of vermis,,rare,,,ZFYVE26,"14:67,728,892-67,816,590",https:/

Would you like to search again?  no


Goodbye!


In [40]:
def test():
    
    # imports
    import pandas as pd # data manipulation/access
    import utils # custom utility functions
    from IPython.display import display, HTML # hyperlink access
    import warnings
    warnings.simplefilter(action='ignore', category=FutureWarning)
    pd.set_option('display.max_columns', None)
    #import fuzzywuzzy # fuzzy match
    #pd.set_option('display.max_colwidth', None)
    #import re # regex

    dimension_1 = ['Diffuse or Multifocal','Symmetric','Frontal vs Posterior Predominance','Telltale Grey Matter Involvement','Cortical Involvement/Subcortical Lesions','U-Fiber/Juxtacortical Involvement', 'Ventriculomegaly vs Hydrocephalus']
    dimension_2 = ['Spinal Cord Involvement','Periventricular Involvement']
    dimension_3 = ['Subcortical Structures']

    # main loop
    flag = True
    while flag:

        # data
        findings_df = pd.read_excel('hsp_data.xlsx').iloc[:, list(range(0,20))].fillna('')
        df = pd.read_excel('hsp_data.xlsx').fillna('')     

        # initially, additional_category is set to True to start a search
        # if set to False, return results with the only selected search category
        additional_category_flag = True

        # keep track of all choices made by the user
        all_choices = []
        while additional_category_flag:
            
            # display list of main categories
            access_categories = list(utils.findings_categorized.keys())
            print("List of categories:")
            print("*******************\n")
            print('\n\n'.join(access_categories))
            print()
            print()

            # user input for main category #
            category = input("Enter name of MRI finding: ")
            
            # keyboard interrupt
            if category == "":
                return "Forced exit."
            
            # invalid input handling #
            while category not in access_categories:
                print()
                category = input("Invalid input. Choose a category from the list above (case sensitive): ")
            
            print()
            print("- You chose: [", category,"].")
            print()
            print()
            
            # check if main category is more than 1 dimension
            if category not in dimension_1:

                # 3 dimension
                if category not in dimension_2:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized['Subcortical Structures']
                    print("List of subcategories:")
                    print("**********************\n")
                    print('\n\n'.join(list(access_subcategories)))
                    print()
                    print()

                    # user input for subcategory #
                    subcategory = input("Choose a subcategory: ")

                    # keyboard interrupt
                    if subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while subcategory not in access_subcategories:
                        print()
                        subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")

                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # display list of sub-subcategories
                    access_sub_subcategories = access_subcategories[subcategory]
                    print("List of sub_subcategories:")
                    print("**************************\n")
                    print('\n\n'.join(list(access_sub_subcategories)))
                    print()
                    print()

                    # user input for sub_subcategory #
                    sub_subcategory = input("Choose a sub-subcategory: ")

                    # keyboard interrupt
                    if sub_subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while sub_subcategory not in access_sub_subcategories:
                        print()
                        sub_subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")
                        
                    print()
                    print("- You chose: [", sub_subcategory,"].")

                    # column title
                    col_title = str(category+':'+subcategory+':'+sub_subcategory)

                # 2 dimension
                else:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized[category]
                    print("\n\nList of subcategories:")
                    print("**********************\n")
                    print('\n\n'.join(list(access_subcategories)))
                    print()
                    print()

                    # user input for subcategory #
                    subcategory = input("Choose a subcategory: ")

                    # keyboard interrupt
                    if subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while subcategory not in access_subcategories:
                        print()
                        subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")

                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # column title
                    col_title = str(category+':'+subcategory)

            # 1 dimension        
            else:

                # column title
                col_title = category
            print()
            print()

            # verify all unique values are selectable
            column_values = list(df[col_title].unique())
            access_values_of_interest = []
            for column_value in column_values:
                values = column_value.split(',')
                for value in values:
                    if str(value.strip()) not in access_values_of_interest:
                        access_values_of_interest.append(value.strip())
                        
            # display list of values of interest
            print("List of Values of Interest:")
            print("***************************\n")
            access_values_of_interest.remove('') if ('' in access_values_of_interest) else None
            print('\n\n'.join(access_values_of_interest))
            print()

            # user input for value of interest #
            value_of_interest = input("Choose a value of interest: ")

            # keyboard interrupt
            if value_of_interest == "":
                return "Forced exit."
            
            # invalid input handling #
            while value_of_interest not in access_values_of_interest:
                print()
                value_of_interest = input("Invalid input. Choose a value of interest from the list above (case sensitive): ")

            print()
            print("- You chose: [", value_of_interest,"].")

            # keep relevant rows
            findings_df = findings_df[findings_df[col_title].str.contains(pat=r'\b{}\b'.format(value_of_interest), regex=True)]

            # update list of user choices #############################################################
            all_choices.append(str(col_title+"%"+value_of_interest))
            
            # prompt user to add additional search constraint #
            add_a_category = input("\n\nWould you to add another category? ")

            # keyboard interrupt
            if add_a_category == "":
                return "Forced exit."

            if add_a_category == '0' or add_a_category.lower()=="no":
                additional_category_flag = False
        print()
        print()
        
        # move "Gene" to first column
        current_columns = df.columns.tolist()
        df = df[['Gene'] + [col for col in current_columns if col != 'Gene']]
        
        # highlight relevant rows
        mask = df.index.isin(findings_df.index.tolist())
        result_rows = findings_df.index.tolist()
        def highlight_rows(row):
            if row.name in result_rows:
                return ['background-color: steelblue'] * len(row)
            else:
                return [''] * len(row)

        # display main results
        result_to_display = pd.DataFrame(df.loc[result_rows]).drop(["Clinical Synopsis"], axis=1)
        results = result_to_display.style.apply(highlight_rows, axis=1)
        display(results)
        print()

        # display rest of df ########################
        rest_of_data = df[~mask]
        rest_of_data["Rank"] = 0
        for i in range(len(all_choices)):
            col = all_choices[i].split('%')[0]
            value = all_choices[i].split('%')[1]
            rest_of_data.loc[rest_of_data[col]==value, 'Rank'] += 1
        rest_of_data = pd.DataFrame(rest_of_data.sort_values(by='Rank', ascending=False))
        display(pd.DataFrame(rest_of_data))

        # prompt user to start new search #
        #print(all_choices)
        terminate = input("Would you like to search again? ")
        if terminate == '0' or terminate.lower() == 'no' or terminate == "":
            flag = False

    # exit message
    print("Goodbye!")

In [ ]:
test_df = test_df.assign(Score = lambda d: fuzz.token_set_ratio("amsterdam",test_df["City"]))

In [50]:
main()

List of categories:
*******************

Diffuse or Multifocal

Symmetric

Frontal vs Posterior Predominance

Telltale Grey Matter Involvement

Cortical Involvement/Subcortical Lesions

U-Fiber/Juxtacortical Involvement

Ventriculomegaly vs Hydrocephalus

Spinal Cord Involvement

Periventricular Involvement

Subcortical Structures




Enter name of MRI finding:  Diffuse or Multifocal



- You chose: [ Diffuse or Multifocal ].




List of Values of Interest:
***************************

D

M



Choose a value of interest:  D



- You chose: [ D ].


...

,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
1,SPG2,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial lesions",pons,present,,,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chrX:103776506-103792619&dgv=pack&knownGene=pack&omimGene=pack,Complicated,XLR,C,"<1/1,000,000","10.3389/fneur.2018.01117 , https://www.ncbi.nlm.nih.gov/books/NBK1509/",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?search=spg2&highlight=spg2,"https://www.orpha.net/consor/cgi-bin/OC_Exp.php?lng=EN&Expert=99015#:~:text=Pure%20SPG2%20manifests%20as%20early,gait%20due%20to%20spastic%20paraparesis.",,Multifocal - MS like lesions
15,SPG47,D,"symmetric, asymmetric",posterior,,"cortical malformation, BPP",,"ear of lynx, ears of grizzly",ears of grizzly,heterotopia,,,present,severe atrophy,severe atrophy,"short CC, hemigenesis",mild atrophy of vermis,,,,ventriculomegaly,AP4B1,"1:113,894,194-113,905,028",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chr1:113894194-113905028&dgv=pack&knownGene=pack&omimGene=pack,Complicated,AR,"I, NN","<1 / 1,000,000","10.3389/fneur.2018.01117 , https://www.ncbi.nlm.nih.gov/books/NBK1509/","10.1093/brain/awz307 , https://doi.org/10.1002/ajmg.a.38561 , 10.1212/WNL.0000000000012836 , https://doi.org/10.1016/j.jns.2011.03.011 , https://doi-org.proxy3.library.mcgill.ca/10.1002/ajmg.a.36514",# 614066 https://www.omim.org/entry/614066?search=spg47&highlight=spg47,https://www.orpha.net/consor/cgi-bin/Disease_Search.php?lng=EN&data_id=20498&Disease_Disease_Search_diseaseGroup=spg47&Disease_Disease_Search_diseaseType=Pat&Disease(s)/group%20of%20diseases=Severe-intellectual-disability-and-progressive-spastic-paraplegia&title=Severe%20intellectual%20disability%20and%20progressive%20spastic%20paraplegia&search=Disease_Search_Simple,,
18,SPG50,D,asymmetric,posterior,,"global white matter atrophy, vascular tortuosity",,present,present,present,,,,present,present,short CC,present,,,,asymmetric ventriculomegaly and CSF issue,AP4M1,"7:100,100,794-100,109,039",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chr7:100100794-100109039&dgv=pack&knownGene=pack&omimGene=pack,Complicated,AR,"I, NN","<1 / 1,000,000","10.3389/fneur.2018.01117 , 10.1002/ajmg.a.36514 , https://www.ncbi.nlm.nih.gov/books/NBK1509/","10.1093/brain/awz307 , 10.1212/WNL.0000000000012836 , https://doi-org.proxy3.library.mcgill.ca/10.1002/ajmg.a.36514 , 10.1016/j.ajhg.2009.06.004 , https://doi.org/10.1212%2FNXG.0000000000000217",# 612936 https://www.omim.org/entry/612936?search=spg50&highlight=spg50,https://www.orpha.net/consor/cgi-bin/Disease_Search.php?lng=EN&data_id=20498&Disease_Disease_Search_diseaseGroup=spg50&Disease_Disease_Search_diseaseType=Pat&Disease(s)/group%20of%20diseases=Severe-intellectual-disability-and-progressive-spastic-paraplegia&title=Severe%20intellec

C:\Users\Aley\AppData\Local\Temp\ipykernel_37572\3331319316.py:225: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rest_of_data["Rank"] = 0


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes,Rank
14,SPG46,D,,posterior,,,,,,present,,,present,present,,,atrophy of vermis,,present,,ventriculomegaly,GBA2,"9:35,736,866-35,749,228",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, C","<1 / 1,000,000",HEAD & NECK\nEars\n- Hearing loss (in some pat...,"10.3389/fneur.2018.01117 , 10.1016/j.ajhg.2012...","https://doi.org/10.1007/s10048-010-0249-2 , 10...",# 614409 https://www.omim.org/entry/614409?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,,2
13,SPG35,D,symmetric,posterior,T2 hypointense pallidal nuclei,,,,ears of grizzly,present,,present,present,present,,,"severe atrophy of cerebellum, atrophy of vermis",,,,ventriculomegaly,FA2H,"16:74,712,969-74,774,820",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"C, AD","<1 / 1,000,000",HEAD & NECK\nEyes\n- External ophthalmoplegia ...,https://www.ncbi.nlm.nih.gov/books/NBK1509/,"10.3389/fneur.2018.01117 , https://doi.org/10....",# 612319 https://www.omim.org/entry/612319?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,,2
7,SPG9B,D,symmetric,posterior,,,,present,present,present,,,,,present,,mild atrophy of vermis,,,,ventriculomegaly,ALDH18A1,"10:95,605,941-95,656,711",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, AD, A","<1 / 1,000,000",GROWTH\n\nHeight\n- Short stature\nOther\n- Gr...,"10.7860/JCDR/2021/49813.15390 , 10.3389/fneur....",https://doi.org/10.1016/j.braindev.2020.07.015,# 616586 https://www.omim.org/entry/616586?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,Not many articles w/ MRI's. Patient is 2.5 yea...,,2
12,SPG21,D,symmetric,posterior,,cortical atrophy,,,present,present,,present,present,present,,,mild atrophy of vermis,,,,,ACP33,"15:64,963,022-64,989,914",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"C, AD, A","<1 / 1,000,000",HEAD & NECK\nMouth\n- Brisk jaw jerk\nABDOMEN\...,"10.3389/fneur.2018.01117 , 10.1055/s-2005-8728...",10.1111/cns.13723,# 248900 https://www.omim.org/entry/248900?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,Also called Mast Syndrome,,2
11,SPG20,D,symmetric,posterior,,corona radiata,,,mild,present,,,,,,,atrophy of superior vermis,,,,ventriculomegaly,SPART,"13:36,301,638-36,370,180",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,A,"<1 / 1,000,000",GROWTH\nHeight\n- Short stature\nHEAD & NECK\n...,https://www.ncbi.nlm.nih.gov/books/NBK1509/,"https://doi.org/10.1007/s11011-017-0104-3 , h...",# 275900 https://www.omim.org/entry/275900?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,Also called Toyer Syndrome (one of the most co...,,2
0,SPG1,D,,,,,,,,,agenesis,agenesis,,,,,,,,,hydrocephalus,L1CAM,"X:153,861,516-153,886,173",https://www.ncbi.nlm.nih.gov/gene/3897,Complicated,XLR,NN,"1/300,000 Male",GROWTH\nHeight\n- Short stature (<5-15th perce...,"https://pubmed.ncbi.nlm.nih.gov/8305964/ , 1

Would you like to search again?  no


Goodbye!
